# 🏗️ Xây dựng hệ thống A2A Multi-Agent với DeepAgents

Chào bạn! Notebook này là một **khóa học thực hành** dạy bạn từng bước xây dựng một hệ thống **A2A multi-agent**:
một **orchestrator** (điều phối viên) chỉ huy **3 agent chuyên môn** bên dưới — mỗi agent được xây bằng framework **DeepAgents**.

## 🎯 Mục tiêu

Sau khóa học này bạn sẽ:

1. Hiểu **A2A (Agent2Agent) là gì** và vì sao các agent cần nói chuyện với nhau theo một "ngôn ngữ chung".
2. Biết các **khối xây dựng cốt lõi** của A2A: `AgentCard`, `Task`, `Message`, `Part`, `Artifact`.
3. Biết cách **đóng gói một DeepAgent thành một A2A Server** (để agent khác gọi được qua mạng).
4. Biết cách dùng **A2A Client** để gọi một agent từ xa.
5. Xây được một **orchestrator** dùng DeepAgents để tự quyết định gọi agent nào.

## 🧠 Giả định về bạn

- Bạn **chưa biết gì về A2A** — tôi sẽ giải thích từ con số 0, kèm ví dụ dễ hiểu như học sinh cấp 3.
- Bạn **đã biết DeepAgents** (`create_deep_agent`, tools, invoke...) — phần này tôi chỉ ôn lại ngắn gọn.

## 🏛️ Kiến trúc hệ thống chúng ta sẽ xây

```mermaid
graph TD
    U([👤 Người dùng]) -->|A2A - gửi câu hỏi| O[🧑‍💼 Orchestrator<br/>DeepAgent + A2A Client]
    O -->|A2A protocol| W[🌤️ Weather Agent<br/>DeepAgent]
    O -->|A2A protocol| N[📰 News Agent<br/>DeepAgent]
    O -->|A2A protocol| C[💱 Currency Agent<br/>DeepAgent]
```

| Thành phần | Nhiệm vụ | Cổng (port) |
|---|---|---|
| `weather_agent.py` | Trả lời thời tiết theo thành phố | 41251 |
| `news_agent.py` | Trả lời tin tức theo chủ đề | 41252 |
| `currency_agent.py` | Chuyển đổi tiền tệ | 41253 |
| `orchestrator_agent.py` | Nghe người dùng, gọi 3 agent trên, tổng hợp kết quả | 41241 |

> 💡 **Mẹo học:** Mỗi agent chạy như **một dịch vụ riêng biệt** (tiến trình riêng, cổng riêng). Đây giống như các công ty khác nhau trên Internet — họ không biết code nội bộ của nhau, chỉ nói chuyện qua **giao thức chuẩn** A2A.


# Chương 1 — A2A là gì? (Giải thích cho người mới)

## 1.1 Vấn đề: các agent "sống cô lập"

Hãy tưởng tượng bạn có 3 "trợ lý tài năng" nhưng mỗi người chỉ nói **một thứ tiếng riêng**:

- Anh **Weather** — giỏi thời tiết, chỉ nói tiếng Anh.
- Chị **News** — giỏi tin tức, chỉ nói tiếng Pháp.
- Anh **Currency** — giỏi tiền tệ, chỉ nói tiếng Nhật.

Muốn họ phối hợp, bạn không thể "nhét" họ vào một cái máy duy nhất (mỗi người có bí quyết riêng, chạy ở công ty riêng). Bạn cần một **ngôn ngữ chung** để họ trao đổi. Đó chính là **A2A Protocol** 🎉

**A2A = Agent2Agent**: một **tiêu chuẩn mở** (open standard) định nghĩa *cách các AI agent nói chuyện với nhau*, bất kể chúng được viết bằng framework nào (LangChain, DeepAgents, ADK, ...) hay thuộc công ty nào.

## 1.2 Ba "diễn viên" chính

```mermaid
graph LR
    U([👤 User]) --> C([🤖 A2A Client])
    C -->|"A2A qua HTTP/JSON-RPC"| S([🤖 A2A Server - Remote Agent])
```

| Diễn viên | Vai trò | Ví dụ trong khóa học |
|---|---|---|
| **User** | Người dùng cuối đưa ra yêu cầu | Bạn, người gõ câu hỏi |
| **A2A Client** | Bên chủ động gửi yêu cầu | Hàm `call_agent()` chúng ta viết; hoặc chính **orchestrator** |
| **A2A Server** | Bên nhận yêu cầu, làm việc, trả kết quả | 3 worker agents + orchestrator |

> 📌 **Điểm mấu chốt:** Một agent thường **vừa là Server** (nhận yêu cầu từ bên trên) **vừa là Client** (gửi yêu cầu xuống bên dưới). Orchestrator của chúng ta chính là "vừa ông chủ, vừa người đi làm".

## 1.3 Các "viên gạch" dữ liệu của A2A

| Khái niệm | Là gì (nói nôm na) | Tên trong code |
|---|---|---|
| **Agent Card** | "Danh thiếp số": agent là ai, giỏi gì, nói chuyện qua đâu | `AgentCard` |
| **Task** | "Phiếu công việc" có số hiệu, có vòng đời (đang làm → xong) | `Task`, `TaskState` |
| **Message** | Một lượt nói chuyện (ai nói, nói gì) | `Message` |
| **Part** | "Phong bì" chứa nội dung (text, file, dữ liệu JSON) | `Part` |
| **Artifact** | "Bàn giao sản phẩm": kết quả cụ thể của Task | `Artifact` |

**Ví dụ analogy — đặt món ở nhà hàng 🍜:**
- **Agent Card** = menu + tên quán: "Quán ABC chuyên món Việt, nhận đặt qua app".
- **Message** = "Cho tôi một tô phở".
- **Task** = phiếu đặt món số #0012 (có thể theo dõi: *đã nhận → đang nấu → hoàn thành*).
- **Part** = nội dung trong phiếu: tô phở, ít hành, thêm ớt.
- **Artifact** = tô phở được bưng ra bàn. 🍜

## 1.4 Vòng đời của một Task

```mermaid
sequenceDiagram
    participant C as Client
    participant S as A2A Server
    C->>S: SendMessage("Thời tiết Hà Nội?")
    S-->>C: Task (submitted) - "đã nhận phiếu"
    S-->>C: status: working - "đang xử lý"
    S-->>C: artifact: "Hà Nội: 28°C, nắng nhẹ"
    S-->>C: status: completed - "hoàn thành"
```

Các trạng thái quan trọng: `submitted` (đã nhận) → `working` (đang làm) → `completed` (xong). Ngoài ra còn có `failed` (lỗi), `canceled` (huỷ), `input-required` (cần hỏi thêm).

## 1.5 Vận chuyển: nói chuyện bằng gì?

A2A chạy trên **HTTP** và nội dung đóng gói theo **JSON-RPC 2.0**. SDK Python hỗ trợ sẵn nhiều "phương tiện vận chuyển":

- **JSON-RPC** (chuẩn, ta dùng cái này)
- **HTTP+JSON (REST)**
- **gRPC** (hiệu năng cao)

Ta không cần tự code JSON-RPC — SDK `a2a-sdk` lo hết. Ta chỉ viết "bộ não" và "danh thiếp".

---

### ✅ Kiểm tra nhanh cài đặt

Chạy cell dưới để chắc chắn `a2a-sdk` đã được cài trong môi trường.


In [1]:
from importlib.metadata import version

for pkg in ["a2a-sdk", "deepagents", "uvicorn"]:
    try:
        print(f"{pkg}: {version(pkg)}")
    except Exception as exc:
        print(f"{pkg}: CHƯA CÀI ({exc})")

a2a-sdk: 1.1.2
deepagents: 0.7.5
uvicorn: 0.35.0


In [2]:
# ! pip install deepagents
# ! pip install langchain-deepseek

# Chương 2 — Chuẩn bị "không gian làm việc"

Ta sẽ đặt toàn bộ code của hệ thống vào thư mục `my_a2a_system/` (nằm cạnh notebook).
Mỗi agent là một file `.py` độc lập — giống như mỗi "công ty" có văn phòng riêng.


In [3]:
import os
import sys

# Thư mục chứa toàn bộ "mã nguồn" của hệ thống A2A
PROJECT_DIR = os.path.join(os.getcwd(), "my_a2a_system")
os.makedirs(PROJECT_DIR, exist_ok=True)

# Để notebook có thể import các hàm tiện ích chung (a2a_common)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

print("Thư mục dự án:", PROJECT_DIR)

Thư mục dự án: c:\Users\tamtt.OFFICEVNPAY\Desktop\A2A\my_a2a_system


# Chương 3 — Chế độ MOCK (học không tốn tiền) ⚙️

**Vấn đề:** Gọi DeepAgent thật cần API key (tốn phí, cần mạng). Nhưng mục tiêu chính của khóa học này là **hiểu A2A** — phần "ống nước" giao tiếp — chứ không phải trí thông minh của agent.

**Giải pháp:** Mỗi agent sẽ có **2 bộ não**:

| Bộ não | Khi nào dùng | Mục đích |
|---|---|---|
| 🧠 **DeepAgent** (thật) | Có API key | Học cách đóng gói DeepAgent vào A2A |
| 🪄 **Mock brain** (giả) | Không có API key | Học phần giao tiếp A2A, chạy nhanh, miễn phí |

> ⚡ **Điểm quan trọng:** Dù dùng bộ não nào, **phần A2A hoàn toàn giống nhau** — client vẫn gọi server qua HTTP, task vẫn có vòng đời. Chỉ khác "nội dung câu trả lời".

### Bật chế độ DeepAgent thật như thế nào?

Set biến môi trường có API key của provider bạn dùng, ví dụ trên Windows PowerShell:

```powershell
$env:ANTHROPIC_API_KEY = "sk-ant-..."
$env:A2A_DEEPAGENT_MODEL = "anthropic:claude-sonnet-5"
```

| Provider | Biến API key | Ví dụ model |
|---|---|---|
| Anthropic | `ANTHROPIC_API_KEY` | `anthropic:claude-sonnet-5` |
| OpenAI | `OPENAI_API_KEY` | `openai:gpt-5.5` |
| Google | `GOOGLE_API_KEY` | `google_genai:gemini-3.5-flash` |

Chương trình sẽ tự dò: **có API key → dùng DeepAgent; không có → tự chuyển sang MOCK**.

### 📦 Viết file tiện ích chung: `a2a_common.py`

Đây là "hộp dụng cụ" dùng chung cho cả 4 agent. Nó chứa 4 việc:

1. `get_model()` — quyết định dùng DeepAgent thật hay mock.
2. `run_deep_agent()` — chạy một DeepAgent, lấy câu trả lời dạng text.
3. `call_agent()` — **chính là A2A Client**: gọi một server từ xa, gom kết quả.
4. `wait_for_server()` — chờ server sẵn sàng (đọc được Agent Card).

Hãy đọc kỹ comment trong code — mỗi dòng đều có giải thích.


In [4]:
# %%writefile {PROJECT_DIR}/a2a_common.py
"""Các hàm tiện ích dùng chung cho hệ thống A2A mini trong bài học."""
import asyncio
import os
import time
import uuid

import httpx

from a2a.client import ClientConfig, create_client
from a2a.helpers import get_artifact_text, get_message_text
from a2a.types import Message, Part, Role, SendMessageRequest, TaskState

# Có .env chứa API key không? Có thì nạp vào.
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

import asyncio


In [5]:
from langchain_deepseek import ChatDeepSeek
from langchain_core.messages import HumanMessage

# Use "deepseek-chat" for DeepSeek-V3 or "deepseek-reasoner" for DeepSeek-R1
llm = ChatDeepSeek(
    model="deepseek-chat",
    temperature=0.7,
    max_tokens=1024
)

# Invoke the model
messages = [HumanMessage(content="Explain quantum computing in one short sentence.")]
response = llm.invoke(messages)

print(response.content)

c:\Users\tamtt.OFFICEVNPAY\AppData\Local\miniconda3\envs\py311ai\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Quantum computing uses the strange rules of quantum physics—like superposition and entanglement—to process information in ways that can solve certain problems exponentially faster than classical computers.


In [6]:

# Model DeepAgents sẽ dùng, định dạng "provider:model".
# Đổi được qua biến môi trường A2A_DEEPAGENT_MODEL.
DEFAULT_MODEL = os.environ.get("A2A_DEEPAGENT_MODEL", "anthropic:claude-sonnet-5")

# Ánh xạ provider -> (các) biến môi trường chứa API key
PROVIDER_KEYS = {
    "anthropic": ("ANTHROPIC_API_KEY",),
    "openai": ("OPENAI_API_KEY",),
    "google_genai": ("GOOGLE_API_KEY", "GEMINI_API_KEY"),
}


def get_model():
    # """Trả về chuỗi model nếu có đủ API key, ngược lại None (chế độ MOCK)."""
    # provider = DEFAULT_MODEL.split(":")[0]
    # key_names = PROVIDER_KEYS.get(provider, ())
    # if any(os.environ.get(k) for k in key_names):
    #     return DEFAULT_MODEL
    # return None
    llm = ChatDeepSeek(
        model="deepseek-chat",
        temperature=0.7,
        max_tokens=10000
    )
    return llm
    

In [7]:
from langchain_core.messages import HumanMessage

In [8]:
# result = LLM.invoke(
#         # {"messages": [{"role": "user", "content": query}]},
#         [HumanMessage(content="HI")],
#         config={"configurable": {"thread_id": "HAKSDF"}},
#     )

In [9]:
async def run_deep_agent(agent, query, thread_id="thread-1"):
    """Chạy agent với câu hỏi, trả về câu trả lời dạng text.

    - DeepAgent (compiled graph): input phải là dict state {"messages": [...]}
      + config thread; result là state dict -> lấy "messages"[-1].content
    - ChatModel thường (như ChatDeepSeek): input là list message; result là
      AIMessage -> lấy .content trực tiếp
    """
    if hasattr(agent, "nodes"):  # DeepAgent (compiled LangGraph)
        result = await agent.ainvoke(
            {"messages": [HumanMessage(content=query)]},
            config={"configurable": {"thread_id": thread_id}},
        )
        content = result["messages"][-1].content
    else:  # ChatModel thường
        result = await agent.ainvoke([HumanMessage(content=query)])
        content = result.content

    # Nội dung có thể là chuỗi đơn giản, hoặc danh sách các "block" (text, image...)
    if isinstance(content, list):
        texts = []
        for block in content:
            if isinstance(block, dict) and block.get("type") == "text":
                texts.append(block.get("text", ""))
        return "\n".join(t for t in texts if t)
    return str(content)


In [10]:
LLM = get_model()
# run_deep_agent(
#     LLM, 'Hi', thread_id='hihi'
# )

In [11]:
from a2a.client import ClientCallContext

# Timeout (giây) cho mỗi lượt đọc stream — đủ lâu cho agent chạy LLM thật.
# Mặc định httpx chỉ 5s nên bị ReadTimeout khi orchestrator mất >5s mới
# sinh sự kiện kế tiếp (đang gọi LLM + nhiều worker qua A2A).
CALL_TIMEOUT = 300.0

# Tiền tố đánh dấu "worker đang cần người dùng xác nhận (HITL)".
# Khi call_agent() không được truyền ask_user, nó trả về chuỗi dạng:
#   "__HITL__::<task_id>|<context_id>|<câu hỏi>"
# để bên gọi (ví dụ orchestrator) tự quyết định cách hỏi người dùng.
HITL_PREFIX = "__HITL__::"


async def call_agent(base_url, user_text, verbose=True, ask_user=None,
                     task_id=None, context_id=None):
    """Gọi một A2A server và trả về toàn bộ nội dung trong artifact.

    Đây chính là "khách hàng" (A2A Client):
      1. Resolve Agent Card từ URL (đọc "danh thiếp")
      2. Gửi SendMessage (truyền task_id/context_id -> gửi TIẾP vào task cũ)
      3. Lắng nghe dòng sự kiện: status update + artifact update
      4. Nếu server chuyển sang input-required (HITL):
         - Có ask_user: gọi ask_user(câu_hỏi) lấy câu trả lời, gửi lại cho
           CÙNG task, tiếp tục tới khi hoàn thành.
         - Không có ask_user: trả về "__HITL__::<task_id>|<context_id>|<câu_hỏi>"
           để bên gọi (orchestrator) tự xử lý việc hỏi người dùng.
    """
    artifacts = []
    current_text = user_text
    cur_task_id = task_id
    cur_context_id = context_id
    async with httpx.AsyncClient() as httpx_client:
        config = ClientConfig(httpx_client=httpx_client)
        client = await create_client(base_url, client_config=config)
        try:
            while True:
                message = Message(
                    role=Role.ROLE_USER,
                    message_id=str(uuid.uuid4()),
                    parts=[Part(text=current_text)],
                    task_id=cur_task_id,
                    context_id=cur_context_id,
                )
                request = SendMessageRequest(message=message)

                question = None  # nếu server cần input (HITL), lưu câu hỏi ở đây
                async for event in client.send_message(
                    request, context=ClientCallContext(timeout=CALL_TIMEOUT)
                ):
                    if event.HasField("task"):
                        cur_task_id = event.task.id
                        cur_context_id = event.task.context_id
                    elif event.HasField("status_update"):
                        su = event.status_update
                        if su.task_id:
                            cur_task_id = su.task_id
                        if su.context_id:
                            cur_context_id = su.context_id
                        state = TaskState.Name(su.status.state)
                        extra = ""
                        if su.status.HasField("message"):
                            extra = get_message_text(
                                su.status.message, delimiter=" "
                            )
                        if verbose:
                            print(f"      [status] {state} {extra}".rstrip())
                        if state == "TASK_STATE_INPUT_REQUIRED":
                            question = extra
                    elif event.HasField("artifact_update"):
                        text = get_artifact_text(
                            event.artifact_update.artifact, delimiter="\n"
                        )
                        if text.strip():
                            artifacts.append(text)

                if question is None:
                    break  # task đã kết thúc -> thoát vòng lặp

                # ---- Server đang cần người dùng (HITL) ----
                if ask_user is None:
                    # Không có cách hỏi người dùng -> trả "dấu hiệu HITL"
                    # kèm id của task để bên gọi tiếp tục sau khi có câu trả lời.
                    return (
                        f"{HITL_PREFIX}{cur_task_id}|{cur_context_id}|{question}"
                    )
                # Hỏi người dùng rồi gửi câu trả lời lại cho CÙNG task.
                # ask_user có thể là hàm thường (trả về str) hoặc async (coroutine).
                reply = ask_user(question)
                if asyncio.iscoroutine(reply):
                    reply = await reply
                current_text = reply
        finally:
            await client.close()
    return "\n".join(artifacts)

# Chương 4 — Bộ khung của một A2A Server 🏗️

Trước khi viết cả 4 agent, ta hiểu **cấu trúc chung** của một A2A Server trong SDK Python.

Mỗi A2A Server gồm **4 mảnh ghép**:

```mermaid
graph LR
    subgraph "Một A2A Server (file .py)"
        A[Agent Card<br/>danh thiếp] --> D[DefaultRequestHandler<br/>lễ tân]
        D --> E[AgentExecutor<br/>bộ não xử lý]
        E --> T[TaskUpdater<br/>phát thanh viên]
    end
```

| Mảnh ghép | Ví von | Vai trò |
|---|---|---|
| `AgentCard` | 🪪 Danh thiếp | Khai báo agent là ai, giỏi gì, nói chuyện qua đâu |
| `DefaultRequestHandler` | 💁 Lễ tân | Nhận yêu cầu từ mạng, quản lý Task, đưa cho bộ não xử lý |
| `AgentExecutor` | 🧠 Bộ não | **Nơi ta viết code**: đọc câu hỏi, suy nghĩ, trả lời |
| `TaskUpdater` | 📢 Phát thanh viên | Báo cho client biết: đang làm, kết quả, hoàn thành |

### 🧠 `AgentExecutor` — nơi ta viết logic

Ta chỉ cần kế thừa lớp `AgentExecutor` và viết 2 hàm:

- `execute(context, event_queue)` — gọi khi có yêu cầu mới. Đọc `context.get_user_input()` để lấy câu hỏi, rồi dùng `TaskUpdater` để báo tiến trình và trả kết quả.
- `cancel(context, event_queue)` — gọi khi client muốn huỷ task.

### 🔄 Luồng xử lý bên trong `execute()`

```mermaid
sequenceDiagram
    participant F as Framework
    participant E as AgentExecutor (của bạn)
    participant U as TaskUpdater
    F->>E: execute(context, event_queue)
    E->>E: đọc câu hỏi: context.get_user_input()
    E->>U: start_work(...)  → báo "đang làm"
    E->>E: suy nghĩ (gọi DeepAgent / mock brain)
    E->>U: add_artifact(...) → giao kết quả
    E->>U: complete() → báo "xong"


# Chương 5 — Viết 3 Worker Agents (DeepAgent + A2A Server) 🌤️📰💱

Mỗi worker là một file `.py` hoàn chỉnh, gồm:

1. **`build_deep_agent()`** — tạo DeepAgent (có tool riêng). Trả về `None` nếu chế độ mock.
2. **`class XxxExecutor(AgentExecutor)`** — bộ não xử lý; bên trong gọi DeepAgent hoặc mock brain.
3. **`create_app()`** — lắp Agent Card + Request Handler + routes thành FastAPI app.
4. **`main()`** — chạy server bằng `uvicorn`.

> 🔍 **Hãy so sánh 3 file:** chúng giống hệt nhau về *khung A2A*; chỉ khác ở *tool* và *mock brain*. Đó chính là triết lý A2A: **"khung giao tiếp chung, trí tuệ riêng"**.


## 5.1 Agent 1 — Weather Agent (`weather_agent.py`) 🌤️

Chạy ở cổng **41251**. Có tool `get_weather(city)`.

```mermaid
graph LR
    U([👤 Người dùng]) --> W[Weather Agent<br/>port 41251]
    W --> T[🛠️ tool: get_weather]
    W --> M[Mock brain]
```


In [12]:
%%writefile {PROJECT_DIR}/weather_agent.py
"""Weather Agent - một A2A Server với "bộ não" là DeepAgent (hoặc mock)."""
import argparse
import logging

import uvicorn
from fastapi import FastAPI

from a2a.server.agent_execution.agent_executor import AgentExecutor
from a2a.server.agent_execution.context import RequestContext
from a2a.server.events.event_queue import EventQueue
from a2a.server.request_handlers import DefaultRequestHandler
from a2a.server.routes import (
    add_a2a_routes_to_fastapi,
    create_agent_card_routes,
    create_jsonrpc_routes,
    create_rest_routes,
)
from a2a.server.tasks.inmemory_task_store import InMemoryTaskStore
from a2a.server.tasks.task_updater import TaskUpdater
from a2a.types import (
    AgentCapabilities,
    AgentCard,
    AgentInterface,
    AgentProvider,
    AgentSkill,
    Part,
    Task,
    TaskState,
    TaskStatus,
)

from a2a_common import get_model, run_deep_agent

logger = logging.getLogger(__name__)


def build_deep_agent():
    """Tạo DeepAgent làm "bộ não" thật. Chưa có API key -> None (dùng mock)."""
    model = get_model()
    if model is None:
        return None

    from deepagents import create_deep_agent
    from langchain.tools import tool

    @tool
    def get_weather(city: str) -> str:
        """Trả về thời tiết hiện tại của một thành phố."""
        table = {
            "hanoi": "Hà Nội: 28°C, trời nắng nhẹ, độ ẩm 75%",
            "hcmc": "TP.HCM: 33°C, nắng nóng, độ ẩm 80%",
            "danang": "Đà Nẵng: 30°C, có mây, khả năng mưa nhẹ",
            "hue": "Huế: 29°C, nắng gián đoạn",
        }
        key = city.strip().lower()
        for k, v in table.items():
            if k in key:
                return v
        return f"{city}: 27°C, trời trong xanh."

    return create_deep_agent(
        name="weather_agent",
        model=model,
        tools=[get_weather],
        system_prompt=(
            "Bạn là chuyên gia thời tiết. Khi được hỏi về thời tiết của một thành phố, "
            "hãy gọi tool get_weather để lấy thông tin rồi trả lời ngắn gọn bằng tiếng Việt."
        ),
    )


class WeatherExecutor(AgentExecutor):
    """Bộ não xử lý mỗi yêu cầu. Framework gọi execute() khi có message mới."""

    def __init__(self, agent):
        self.agent = agent

    async def execute(self, context: RequestContext, event_queue: EventQueue):
        # 1) Đọc câu hỏi người dùng
        user_input = context.get_user_input()
        task_id = context.task_id
        context_id = context.context_id

        # 2) BẮT BUỘC: gửi Task ban đầu (submitted) TRƯỚC các status update
        await event_queue.enqueue_event(
            Task(
                id=task_id,
                context_id=context_id,
                status=TaskStatus(state=TaskState.TASK_STATE_SUBMITTED),
                history=[context.message],
            )
        )

        # 3) "Phát thanh viên" báo trạng thái cho client
        updater = TaskUpdater(event_queue, task_id, context_id)
        await updater.start_work(
            message=updater.new_agent_message(
                parts=[Part(text="Đang kiểm tra thời tiết...")]
            )
        )

        # 4) Suy nghĩ: DeepAgent thật HOẶC mock brain
        answer = await run_deep_agent(self.agent, user_input, thread_id=task_id)

        # 5) Giao kết quả (artifact) và báo hoàn thành
        await updater.add_artifact(
            parts=[Part(text=answer)], name="response", last_chunk=True
        )
        await updater.complete()

    def mock_brain(self, query: str) -> str:
        """Bộ não giả - không cần LLM, chỉ để học phần giao tiếp A2A."""
        cities = {"hanoi": "Hà Nội", "hcmc": "TP.HCM", "danang": "Đà Nẵng", "hue": "Huế"}
        q = query.lower()
        for key, name in cities.items():
            if key in q:
                return f"[MOCK weather] {name}: 28°C, trời nắng nhẹ."
        return "[MOCK weather] 27°C, trời trong xanh."

    async def cancel(self, context: RequestContext, event_queue: EventQueue):
        updater = TaskUpdater(event_queue, context.task_id, context.context_id)
        await updater.cancel()


def create_app(host: str, port: int):
    """Lắp ráp server: Agent Card + Request Handler + routes."""
    # Agent Card = "danh thiếp số" của agent
    agent_card = AgentCard(
        name="Weather Agent",
        description="Chuyên gia thời tiết: trả lời câu hỏi về thời tiết theo thành phố.",
        provider=AgentProvider(organization="A2A Course", url="http://example.com"),
        version="1.0.0",
        capabilities=AgentCapabilities(streaming=True, push_notifications=False),
        default_input_modes=["text"],
        default_output_modes=["text", "task-status"],
        skills=[
            AgentSkill(
                id="weather",
                name="Weather lookup",
                description="Hỏi thời tiết của một thành phố.",
                tags=["weather"],
                examples=["Thời tiết Hà Nội thế nào?"],
                input_modes=["text"],
                output_modes=["text", "task-status"],
            )
        ],
        supported_interfaces=[
            AgentInterface(
                protocol_binding="JSONRPC",
                protocol_version="1.0",
                url=f"http://{host}:{port}/a2a/jsonrpc",
            )
        ],
    )

    request_handler = DefaultRequestHandler(
        agent_executor=WeatherExecutor(build_deep_agent()),
        task_store=InMemoryTaskStore(),
        agent_card=agent_card,
    )

    app = FastAPI(title=agent_card.name)
    add_a2a_routes_to_fastapi(
        app,
        agent_card_routes=create_agent_card_routes(agent_card=agent_card),
        jsonrpc_routes=create_jsonrpc_routes(
            request_handler=request_handler,
            rpc_url="/a2a/jsonrpc",
            enable_v0_3_compat=True,
        ),
        rest_routes=create_rest_routes(
            request_handler=request_handler,
            path_prefix="/a2a/rest",
            enable_v0_3_compat=True,
        ),
    )
    return app


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--host", default="127.0.0.1")
    parser.add_argument("--port", type=int, default=41251)
    args = parser.parse_args()
    logging.basicConfig(level=logging.INFO)
    uvicorn.run(create_app(args.host, args.port), host=args.host, port=args.port)


if __name__ == "__main__":
    main()


Overwriting c:\Users\tamtt.OFFICEVNPAY\Desktop\A2A\my_a2a_system/weather_agent.py


## 5.2 Agent 2 — News Agent (`news_agent.py`) 📰

Chạy ở cổng **41252**. Có tool `get_news(topic)`.

> 👀 **Thử thách nhỏ:** đối chiếu file này với `weather_agent.py` — bạn sẽ thấy **chỉ khác 3 chỗ**: tên tool, nội dung mock brain, và tên lớp/port. Phần khung A2A (executor, agent card, routes) **giống hệt**.


In [13]:
%%writefile {PROJECT_DIR}/news_agent.py
"""News Agent - trả lời tin tức theo chủ đề."""
import argparse
import logging

import uvicorn
from fastapi import FastAPI

from a2a.server.agent_execution.agent_executor import AgentExecutor
from a2a.server.agent_execution.context import RequestContext
from a2a.server.events.event_queue import EventQueue
from a2a.server.request_handlers import DefaultRequestHandler
from a2a.server.routes import (
    add_a2a_routes_to_fastapi,
    create_agent_card_routes,
    create_jsonrpc_routes,
    create_rest_routes,
)
from a2a.server.tasks.inmemory_task_store import InMemoryTaskStore
from a2a.server.tasks.task_updater import TaskUpdater
from a2a.types import (
    AgentCapabilities,
    AgentCard,
    AgentInterface,
    AgentProvider,
    AgentSkill,
    Part,
    Task,
    TaskState,
    TaskStatus,
)

from a2a_common import get_model, run_deep_agent

logger = logging.getLogger(__name__)


def build_deep_agent():
    """Tạo DeepAgent làm "bộ não" thật. Chưa có API key -> None (dùng mock)."""
    model = get_model()
    if model is None:
        return None

    from deepagents import create_deep_agent
    from langchain.tools import tool

    @tool
    def get_news(topic: str) -> str:
        """Trả về các dòng tít tin tức nổi bật về một chủ đề."""
        topic = topic.strip().lower()
        headlines = {
            "công nghệ": [
                "AI thay đổi cách lập trình viên làm việc",
                "Ra mắt chip điện toán lượng tử mới",
            ],
            "kinh tế": [
                "GDP quý này tăng trưởng 6.5%",
                "Giá vàng lập đỉnh mới",
            ],
            "thể thao": [
                "Đội tuyển quốc gia giành chiến thắng 2-0",
                "Giải bóng đá mở màn sôi động",
            ],
        }
        for k, v in headlines.items():
            if k in topic:
                return "\n".join(f"- {h}" for h in v)
        return f"- Tin nóng về {topic}\n- Phân tích chuyên sâu về {topic}"

    return create_deep_agent(
        name="news_agent",
        model=model,
        tools=[get_news],
        system_prompt=(
            "Bạn là chuyên gia tin tức. Khi được hỏi về tin tức theo chủ đề, "
            "hãy gọi tool get_news rồi trả lời ngắn gọn bằng tiếng Việt."
        ),
    )


class NewsExecutor(AgentExecutor):
    def __init__(self, agent):
        self.agent = agent

    async def execute(self, context: RequestContext, event_queue: EventQueue):
        user_input = context.get_user_input()
        task_id = context.task_id
        context_id = context.context_id

        # BẮT BUỘC: gửi Task ban đầu (submitted) TRƯỚC các status update
        await event_queue.enqueue_event(
            Task(
                id=task_id,
                context_id=context_id,
                status=TaskStatus(state=TaskState.TASK_STATE_SUBMITTED),
                history=[context.message],
            )
        )

        updater = TaskUpdater(event_queue, task_id, context_id)
        await updater.start_work(
            message=updater.new_agent_message(
                parts=[Part(text="Đang tìm tin tức...")]
            )
        )

        answer = await run_deep_agent(self.agent, user_input, thread_id=task_id)
        await updater.add_artifact(
            parts=[Part(text=answer)], name="response", last_chunk=True
        )
        await updater.complete()

    def mock_brain(self, query: str) -> str:
        q = query.lower()
        topics = {"công nghệ": "công nghệ", "tech": "công nghệ",
                  "kinh tế": "kinh tế", "thể thao": "thể thao"}
        for key, label in topics.items():
            if key in q:
                return (
                    f"[MOCK news] Tin nổi bật về {label}:\n"
                    f"- Tiêu đề số 1 về {label}\n"
                    f"- Tiêu đề số 2 về {label}"
                )
        return "[MOCK news] Tin nóng: Hôm nay thị trường khởi sắc, trời nắng đẹp."

    async def cancel(self, context: RequestContext, event_queue: EventQueue):
        updater = TaskUpdater(event_queue, context.task_id, context.context_id)
        await updater.cancel()


def create_app(host: str, port: int):
    agent_card = AgentCard(
        name="News Agent",
        description="Chuyên gia tin tức: trả lời các dòng tít tin tức theo chủ đề.",
        provider=AgentProvider(organization="A2A Course", url="http://example.com"),
        version="1.0.0",
        capabilities=AgentCapabilities(streaming=True, push_notifications=False),
        default_input_modes=["text"],
        default_output_modes=["text", "task-status"],
        skills=[
            AgentSkill(
                id="news",
                name="News lookup",
                description="Hỏi tin tức theo chủ đề.",
                tags=["news"],
                examples=["Tin tức công nghệ hôm nay?"],
                input_modes=["text"],
                output_modes=["text", "task-status"],
            )
        ],
        supported_interfaces=[
            AgentInterface(
                protocol_binding="JSONRPC",
                protocol_version="1.0",
                url=f"http://{host}:{port}/a2a/jsonrpc",
            )
        ],
    )

    request_handler = DefaultRequestHandler(
        agent_executor=NewsExecutor(build_deep_agent()),
        task_store=InMemoryTaskStore(),
        agent_card=agent_card,
    )

    app = FastAPI(title=agent_card.name)
    add_a2a_routes_to_fastapi(
        app,
        agent_card_routes=create_agent_card_routes(agent_card=agent_card),
        jsonrpc_routes=create_jsonrpc_routes(
            request_handler=request_handler,
            rpc_url="/a2a/jsonrpc",
            enable_v0_3_compat=True,
        ),
        rest_routes=create_rest_routes(
            request_handler=request_handler,
            path_prefix="/a2a/rest",
            enable_v0_3_compat=True,
        ),
    )
    return app


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--host", default="127.0.0.1")
    parser.add_argument("--port", type=int, default=41252)
    args = parser.parse_args()
    logging.basicConfig(level=logging.INFO)
    uvicorn.run(create_app(args.host, args.port), host=args.host, port=args.port)


if __name__ == "__main__":
    main()


Overwriting c:\Users\tamtt.OFFICEVNPAY\Desktop\A2A\my_a2a_system/news_agent.py


## 5.3 Agent 3 — Currency Agent (`currency_agent.py`) 💱

Chạy ở cổng **41253**. Có tool `convert_currency(amount, from_ccy, to_ccy)`.


In [14]:
%%writefile {PROJECT_DIR}/currency_agent.py
"""Currency Agent - chuyển đổi tiền tệ."""
import argparse
import logging
import re

import uvicorn
from fastapi import FastAPI

from a2a.server.agent_execution.agent_executor import AgentExecutor
from a2a.server.agent_execution.context import RequestContext
from a2a.server.events.event_queue import EventQueue
from a2a.server.request_handlers import DefaultRequestHandler
from a2a.server.routes import (
    add_a2a_routes_to_fastapi,
    create_agent_card_routes,
    create_jsonrpc_routes,
    create_rest_routes,
)
from a2a.server.tasks.inmemory_task_store import InMemoryTaskStore
from a2a.server.tasks.task_updater import TaskUpdater
from a2a.types import (
    AgentCapabilities,
    AgentCard,
    AgentInterface,
    AgentProvider,
    AgentSkill,
    Part,
    Task,
    TaskState,
    TaskStatus,
)

from a2a_common import get_model, run_deep_agent

logger = logging.getLogger(__name__)

# Tỷ giá giả lập (đơn vị: 1 ngoại tệ = X VND)
RATES = {"USD": 24500, "EUR": 26500, "GBP": 31000, "JPY": 165, "VND": 1}


def build_deep_agent():
    """Tạo DeepAgent làm "bộ não" thật. Chưa có API key -> None (dùng mock)."""
    model = get_model()
    if model is None:
        return None

    from deepagents import create_deep_agent
    from langchain.tools import tool

    @tool
    def convert_currency(amount: float, from_ccy: str, to_ccy: str) -> str:
        """Chuyển một số tiền từ đơn vị tiền tệ này sang đơn vị khác."""
        f = from_ccy.strip().upper()
        t = to_ccy.strip().upper()
        if f not in RATES or t not in RATES:
            return "Không tìm thấy tỷ giá cho cặp tiền này."
        result = amount * RATES[f] / RATES[t]
        return (
            f"{amount:,.0f} {f} = {result:,.0f} {t} "
            f"(tỷ giá 1 {f} = {RATES[f] / RATES[t]:,.2f} {t})"
        )

    return create_deep_agent(
        name="currency_agent",
        model=model,
        tools=[convert_currency],
        system_prompt=(
            "Bạn là chuyên gia tài chính. Khi được hỏi về chuyển đổi tiền tệ, "
            "hãy gọi tool convert_currency (nhớ tách số tiền và 2 đơn vị tiền) "
            "rồi trả lời ngắn gọn bằng tiếng Việt."
        ),
    )


class CurrencyExecutor(AgentExecutor):
    def __init__(self, agent):
        self.agent = agent

    async def execute(self, context: RequestContext, event_queue: EventQueue):
        user_input = context.get_user_input()
        task_id = context.task_id
        context_id = context.context_id

        # BẮT BUỘC: gửi Task ban đầu (submitted) TRƯỚC các status update
        await event_queue.enqueue_event(
            Task(
                id=task_id,
                context_id=context_id,
                status=TaskStatus(state=TaskState.TASK_STATE_SUBMITTED),
                history=[context.message],
            )
        )

        updater = TaskUpdater(event_queue, task_id, context_id)
        await updater.start_work(
            message=updater.new_agent_message(
                parts=[Part(text="Đang tra tỷ giá...")]
            )
        )

        answer = await run_deep_agent(self.agent, user_input, thread_id=task_id)

        await updater.add_artifact(
            parts=[Part(text=answer)], name="response", last_chunk=True
        )
        await updater.complete()

    async def cancel(self, context: RequestContext, event_queue: EventQueue):
        updater = TaskUpdater(event_queue, context.task_id, context.context_id)
        await updater.cancel()


def create_app(host: str, port: int):
    agent_card = AgentCard(
        name="Currency Agent",
        description="Chuyên gia tài chính: chuyển đổi tiền tệ giữa các đơn vị.",
        provider=AgentProvider(organization="A2A Course", url="http://example.com"),
        version="1.0.0",
        capabilities=AgentCapabilities(streaming=True, push_notifications=False),
        default_input_modes=["text"],
        default_output_modes=["text", "task-status"],
        skills=[
            AgentSkill(
                id="currency",
                name="Currency conversion",
                description="Đổi tiền giữa các đơn vị tiền tệ.",
                tags=["currency"],
                examples=["100 USD bằng bao nhiêu VND?"],
                input_modes=["text"],
                output_modes=["text", "task-status"],
            )
        ],
        supported_interfaces=[
            AgentInterface(
                protocol_binding="JSONRPC",
                protocol_version="1.0",
                url=f"http://{host}:{port}/a2a/jsonrpc",
            )
        ],
    )

    request_handler = DefaultRequestHandler(
        agent_executor=CurrencyExecutor(build_deep_agent()),
        task_store=InMemoryTaskStore(),
        agent_card=agent_card,
    )

    app = FastAPI(title=agent_card.name)
    add_a2a_routes_to_fastapi(
        app,
        agent_card_routes=create_agent_card_routes(agent_card=agent_card),
        jsonrpc_routes=create_jsonrpc_routes(
            request_handler=request_handler,
            rpc_url="/a2a/jsonrpc",
            enable_v0_3_compat=True,
        ),
        rest_routes=create_rest_routes(
            request_handler=request_handler,
            path_prefix="/a2a/rest",
            enable_v0_3_compat=True,
        ),
    )
    return app


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--host", default="127.0.0.1")
    parser.add_argument("--port", type=int, default=41253)
    args = parser.parse_args()
    logging.basicConfig(level=logging.INFO)
    uvicorn.run(create_app(args.host, args.port), host=args.host, port=args.port)


if __name__ == "__main__":
    main()


Overwriting c:\Users\tamtt.OFFICEVNPAY\Desktop\A2A\my_a2a_system/currency_agent.py


# Chương 6 — Khởi động các server 🚀

### Vì sao dùng `subprocess`?

Mỗi agent là một **dịch vụ độc lập** (file `.py` riêng). Trong notebook, ta không chạy chúng trực tiếp trong cùng tiến trình (sẽ "kẹt" vì server chạy vô hạn). Thay vào đó ta **mở một tiến trình con** cho mỗi agent — giống như mở 4 "cửa hàng" riêng biệt trên 4 con phố khác nhau.

Hàm `start_server()` dưới đây:

1. Mở tiến trình `python <file>.py --port <port>` (dùng chính Python của kernel).
2. Ghi log ra file `<tên>.log` (để tra cứu khi gặp lỗi).
3. Lưu lại "điều khiển" tiến trình để sau này tắt được.


In [15]:
import subprocess
import sys

PROCESSES = []  # danh sách "điều khiển" các server đang chạy


def start_server(name, port):
    """Mở một agent server trong tiến trình con."""
    script = os.path.join(PROJECT_DIR, f"{name}.py")
    log_path = os.path.join(PROJECT_DIR, f"{name}.log")
    log_file = open(log_path, "w", encoding="utf-8")
    proc = subprocess.Popen(
        [sys.executable, script, "--port", str(port)],
        stdout=log_file,
        stderr=subprocess.STDOUT,
        cwd=PROJECT_DIR,
    )
    PROCESSES.append(proc)
    print(f"Đã khởi động {name} (pid={proc.pid}) -> log: {os.path.basename(log_path)}")
    return proc


def stop_all_servers():
    """Tắt tất cả server (gọi khi kết thúc buổi học)."""
    for proc in PROCESSES:
        proc.terminate()
    for proc in PROCESSES:
        try:
            proc.wait(timeout=5)
        except Exception:
            proc.kill()
    PROCESSES.clear()
    print("Đã dừng tất cả server.")

In [16]:
# Khởi động 3 worker agents
start_server("weather_agent", 41251)
start_server("news_agent", 41252)
start_server("currency_agent", 41253)

Đã khởi động weather_agent (pid=14456) -> log: weather_agent.log
Đã khởi động news_agent (pid=37952) -> log: news_agent.log
Đã khởi động currency_agent (pid=18480) -> log: currency_agent.log


<Popen: returncode: None args: ['c:\\Users\\tamtt.OFFICEVNPAY\\AppData\\Loca...>

### ⏳ Chờ server sẵn sàng

Server cần vài giây để "dọn hàng" (import thư viện, mở cổng). Ta chờ tới khi đọc được **Agent Card** — giống như chờ cửa hàng treo biển "đã mở cửa" 🏪.


In [17]:
from a2a_common import wait_for_server

# Chờ cả 3 worker sẵn sàng
for name, port in [("weather", 41251), ("news", 41252), ("currency", 41253)]:
    url = f"http://127.0.0.1:{port}"
    ok = await wait_for_server(url)
    print(f"{name:10s} {url:35s} -> {'SẴN SÀNG ✅' if ok else 'LỖI ❌'}")

weather    http://127.0.0.1:41251              -> SẴN SÀNG ✅
news       http://127.0.0.1:41252              -> SẴN SÀNG ✅
currency   http://127.0.0.1:41253              -> SẴN SÀNG ✅


# Chương 7 — A2A Client: gọi agent từ xa 📞

Giờ ta đóng vai **User** để gọi từng worker. Hàm `call_agent()` trong `a2a_common.py` chính là A2A Client.

### Điều gì xảy ra khi gọi?

```mermaid
sequenceDiagram
    participant N as Notebook (Client)
    participant W as Weather Agent (Server)
    N->>W: GET /.well-known/agent-card.json (đọc danh thiếp)
    W-->>N: AgentCard
    N->>W: SendMessage("Thời tiết Hà Nội?")
    W-->>N: status: submitted
    W-->>N: status: working
    W-->>N: artifact: "Hà Nội 28°C..."
    W-->>N: status: completed
```

> 💡 Chú ý dòng `[status] ...` in ra: đó là **TaskUpdater bên server "phát thanh"** về cho client. Client nhận được sự kiện qua **stream** (SSE). Đây là sức mạnh của A2A: ta luôn biết task đang ở đâu trong vòng đời.


In [18]:
# Gọi trực tiếp Weather Agent
ket_qua = await call_agent(
    "http://127.0.0.1:41251", "Thời tiết Hà Nội hôm nay thế nào?")
print()
print("=== KẾT QUẢ WEATHER ===")
print(ket_qua)

      [status] TASK_STATE_WORKING Đang kiểm tra thời tiết...
      [status] TASK_STATE_COMPLETED

=== KẾT QUẢ WEATHER ===
Hôm nay thời tiết Hà Nội rất đẹp! Trời trong xanh, nhiệt độ khoảng 27°C, thời tiết dễ chịu, thích hợp để ra ngoài hoạt động. Bạn nhớ mang theo nước uống và bảo vệ da khi ra nắng nhé! ☀️


In [19]:
# Gọi trực tiếp News Agent và Currency Agent
print("--- NEWS ---")
print(await call_agent("http://127.0.0.1:41252", "Cho tôi tin tức công nghệ hôm nay"))
print()
print("--- CURRENCY ---")
print(await call_agent("http://127.0.0.1:41253", "100 USD bằng bao nhiêu VND?"))

--- NEWS ---
      [status] TASK_STATE_WORKING Đang tìm tin tức...
      [status] TASK_STATE_COMPLETED
Dưới đây là tin tức công nghệ nổi bật hôm nay:

1. **AI thay đổi cách lập trình viên làm việc** — Trí tuệ nhân tạo đang tạo ra những thay đổi lớn trong quy trình và cách thức làm việc của lập trình viên, từ việc hỗ trợ viết code đến tối ưu hóa quy trình phát triển phần mềm.

2. **Ra mắt chip điện toán lượng tử mới** — Một thế hệ chip điện toán lượng tử mới vừa được công bố, hứa hẹn mang lại những bước tiến vượt bậc về khả năng tính toán trong tương lai.

Nếu bạn muốn tìm hiểu sâu hơn về chủ đề nào, hãy cho tôi biết nhé!

--- CURRENCY ---
      [status] TASK_STATE_WORKING Đang tra tỷ giá...
      [status] TASK_STATE_COMPLETED
100 USD tương đương **2.450.000 VND** (theo tỷ giá 1 USD = 24.500 VND).


# Chương 8 — Orchestrator: "đội trưởng" 🧑‍💼

Đến đây ta đã có 3 chuyên gia độc lập. Vấn đề: **ai là người nghe người dùng và quyết định gọi ai?**

Đó chính là **orchestrator** — một DeepAgent đặc biệt:

- Nó là **A2A Server** (để người dùng nói chuyện với nó qua A2A).
- Bên trong là **DeepAgent** với 3 tools: `ask_weather`, `ask_news`, `ask_currency`.
- Mỗi tool, khi được gọi, sẽ dùng **A2A Client** (`call_agent`) để gọi worker tương ứng ở xa.

```mermaid
sequenceDiagram
    participant U as 👤 User
    participant O as 🧑‍💼 Orchestrator (DeepAgent)
    participant W as 🌤️ Weather Agent
    participant C as 💱 Currency Agent
    U->>O: "Thời tiết HN thế nào và 100 USD = ?"
    O->>O: DeepAgent quyết định gọi tool nào
    O->>W: call_agent(weather) - A2A
    W-->>O: "Hà Nội 28°C..."
    O->>C: call_agent(currency) - A2A
    C-->>O: "100 USD = 2.450.000 VND"
    O-->>U: Tổng hợp thành 1 câu trả lời
```

> 🎯 **Bản chất quan trọng:** Orchestrator **không trực tiếp tính** thời tiết hay tỷ giá. Nó **hỏi** các chuyên gia qua A2A rồi gom kết quả. Đúng như một người quản lý giỏi: *không tự làm, mà biết giao đúng người*.

> 🔁 **Vòng lặp "vừa Server vừa Client":** Orchestrator nhận yêu cầu với vai **Server**, rồi gọi worker với vai **Client**. Vai trò đảo nhau tuỳ hướng nói chuyện.


In [20]:
%%writefile {PROJECT_DIR}/orchestrator_agent.py
"""Orchestrator Agent - điều phối các worker agents qua giao thức A2A.

Vừa là A2A Server (nhận yêu cầu từ User) vừa là A2A Client (gọi worker).
Hỗ trợ HUMAN-IN-THE-LOOP (HITL) DẠNG VÒNG LẶP: nếu một worker (vd Booking Agent)
cần người dùng xác nhận, orchestrator chuyển tiếp câu hỏi lên người dùng bằng
trạng thái input-required. Mỗi lần người dùng trả lời CHƯA xác nhận, worker hỏi
lại -> orchestrator LẠI chuyển tiếp câu hỏi mới lên người dùng... lặp cho tới khi
người dùng xác nhận thật sự và worker hoàn tất.
"""
import argparse
import logging
import os

import uvicorn
from fastapi import FastAPI

from a2a.server.agent_execution.agent_executor import AgentExecutor
from a2a.server.agent_execution.context import RequestContext
from a2a.server.events.event_queue import EventQueue
from a2a.server.request_handlers import DefaultRequestHandler
from a2a.server.routes import (
    add_a2a_routes_to_fastapi,
    create_agent_card_routes,
    create_jsonrpc_routes,
    create_rest_routes,
)
from a2a.server.tasks.inmemory_task_store import InMemoryTaskStore
from a2a.server.tasks.task_updater import TaskUpdater
from a2a.types import (
    AgentCapabilities,
    AgentCard,
    AgentInterface,
    AgentProvider,
    AgentSkill,
    Part,
    Task,
    TaskState,
    TaskStatus,
)

from a2a_common import HITL_PREFIX, call_agent, get_model, run_deep_agent

logger = logging.getLogger(__name__)

# Địa chỉ 4 worker (có thể đổi qua biến môi trường)
WEATHER_URL = os.environ.get("A2A_WEATHER_URL", "http://127.0.0.1:41251")
NEWS_URL = os.environ.get("A2A_NEWS_URL", "http://127.0.0.1:41252")
CURRENCY_URL = os.environ.get("A2A_CURRENCY_URL", "http://127.0.0.1:41253")
BOOKING_URL = os.environ.get("A2A_BOOKING_URL", "http://127.0.0.1:41254")


def build_deep_agent():
    """DeepAgent làm "bộ não điều phối". Chưa có API key -> None (dùng mock)."""
    model = get_model()
    if model is None:
        return None

    from deepagents import create_deep_agent
    from langchain.tools import tool

    # Mỗi tool = một "đường dây điện thoại" tới một worker qua A2A
    @tool
    async def ask_weather(query: str) -> str:
        """Gửi câu hỏi về thời tiết cho Weather Agent."""
        return await call_agent(WEATHER_URL, query, verbose=False)

    @tool
    async def ask_news(query: str) -> str:
        """Gửi câu hỏi về tin tức cho News Agent."""
        return await call_agent(NEWS_URL, query, verbose=False)

    @tool
    async def ask_currency(query: str) -> str:
        """Gửi câu hỏi về chuyển đổi tiền tệ cho Currency Agent."""
        return await call_agent(CURRENCY_URL, query, verbose=False)

    @tool
    async def ask_booking(query: str) -> str:
        """Gửi yêu cầu đặt vé máy bay cho Booking Agent."""
        return await call_agent(BOOKING_URL, query, verbose=False)

    return create_deep_agent(
        name="orchestrator",
        model=model,
        tools=[ask_weather, ask_news, ask_currency, ask_booking],
        system_prompt=(
            "Bạn là đội trưởng điều phối. Quy tắc:\n"
            "- Hỏi về thời tiết -> gọi ask_weather\n"
            "- Hỏi về tin tức -> gọi ask_news\n"
            "- Hỏi về tiền tệ/đổi tiền -> gọi ask_currency\n"
            "- Hỏi về đặt vé máy bay -> gọi ask_booking\n"
            "- Yêu cầu phức tạp -> gọi NHIỀU tool và tổng hợp.\n"
            "- Nếu một tool trả về chuỗi bắt đầu bằng "
            f"'{HITL_PREFIX}', hãy trả lời NGUYÊN VĂN chuỗi đó làm câu "
            "trả lời cuối cùng, KHÔNG thêm, bớt hay diễn giải gì cả.\n"
            "Cuối cùng hãy trả lời rõ ràng, thân thiện bằng tiếng Việt."
        ),
    )


class OrchestratorExecutor(AgentExecutor):
    def __init__(self, agent):
        self.agent = agent
        # task_id (của orchestrator) -> thông tin worker đang chờ xác nhận (HITL)
        self.pending = {}

    async def execute(self, context: RequestContext, event_queue: EventQueue):
        user_input = context.get_user_input()
        task_id = context.task_id
        context_id = context.context_id
        updater = TaskUpdater(event_queue, task_id, context_id)

        # ============ LẦN GỌI THỨ 2+ (RESUME): người dùng vừa trả lời ============
        if task_id in self.pending:
            worker_url, w_task_id, w_ctx_id = self.pending.pop(task_id)
            await updater.start_work(
                message=updater.new_agent_message(
                    parts=[Part(text="Đang chuyển câu trả lời của bạn tới chuyên gia...")]
                )
            )
            # Chuyển câu trả lời xuống CÙNG task của worker để worker tiếp tục
            result = await call_agent(
                worker_url, user_input, verbose=False,
                task_id=w_task_id, context_id=w_ctx_id,
            )

            # Worker vẫn cần hỏi thêm (HITL lặp) -> chuyển tiếp câu hỏi mới
            # lên người dùng và tiếp tục chờ (vòng lặp) cho tới khi hoàn tất.
            if isinstance(result, str) and result.startswith(HITL_PREFIX):
                payload = result[len(HITL_PREFIX):].strip()
                w_task_id, w_ctx_id, question = payload.split("|", 2)
                self.pending[task_id] = (worker_url, w_task_id, w_ctx_id)
                await updater.requires_input(
                    message=updater.new_agent_message(parts=[Part(text=question)])
                )
                return

            # Worker đã hoàn tất -> trả kết quả cuối
            await updater.add_artifact(
                parts=[Part(text=result)], name="response", last_chunk=True
            )
            await updater.complete()
            return

        # ============ LẦN GỌI ĐẦU: chạy DeepAgent để chọn worker ============
        # BẮT BUỘC: gửi Task ban đầu (submitted) TRƯỚC các status update
        await event_queue.enqueue_event(
            Task(
                id=task_id,
                context_id=context_id,
                status=TaskStatus(state=TaskState.TASK_STATE_SUBMITTED),
                history=[context.message],
            )
        )

        await updater.start_work(
            message=updater.new_agent_message(
                parts=[Part(text="Đang điều phối các chuyên gia...")]
            )
        )

        answer = await run_deep_agent(self.agent, user_input, thread_id=task_id)

        # Một worker (vd Booking Agent) cần người dùng xác nhận: DeepAgent trả
        # về đúng "dấu hiệu HITL" -> orchestrator chuyển task sang input-required.
        if isinstance(answer, str) and answer.startswith(HITL_PREFIX):
            payload = answer[len(HITL_PREFIX):].strip()
            w_task_id, w_ctx_id, question = payload.split("|", 2)
            self.pending[task_id] = (BOOKING_URL, w_task_id, w_ctx_id)
            await updater.requires_input(
                message=updater.new_agent_message(parts=[Part(text=question)])
            )
            return  # nhường quyền: chờ người dùng trả lời ở lần gọi sau

        await updater.add_artifact(
            parts=[Part(text=answer)], name="response", last_chunk=True
        )
        await updater.complete()

    async def cancel(self, context: RequestContext, event_queue: EventQueue):
        updater = TaskUpdater(event_queue, context.task_id, context.context_id)
        await updater.cancel()


def create_app(host: str, port: int):
    agent_card = AgentCard(
        name="Orchestrator Agent",
        description="Điều phối viên: nghe yêu cầu, gọi các chuyên gia (weather/news/currency/booking) và tổng hợp.",
        provider=AgentProvider(organization="A2A Course", url="http://example.com"),
        version="1.0.0",
        capabilities=AgentCapabilities(streaming=True, push_notifications=False),
        default_input_modes=["text"],
        default_output_modes=["text", "task-status"],
        skills=[
            AgentSkill(
                id="orchestrate",
                name="Orchestrate specialists",
                description="Điều phối các agent chuyên môn.",
                tags=["orchestration"],
                examples=["Thời tiết Hà Nội và giá USD hôm nay?"],
                input_modes=["text"],
                output_modes=["text", "task-status"],
            )
        ],
        supported_interfaces=[
            AgentInterface(
                protocol_binding="JSONRPC",
                protocol_version="1.0",
                url=f"http://{host}:{port}/a2a/jsonrpc",
            )
        ],
    )

    request_handler = DefaultRequestHandler(
        agent_executor=OrchestratorExecutor(build_deep_agent()),
        task_store=InMemoryTaskStore(),
        agent_card=agent_card,
    )

    app = FastAPI(title=agent_card.name)
    add_a2a_routes_to_fastapi(
        app,
        agent_card_routes=create_agent_card_routes(agent_card=agent_card),
        jsonrpc_routes=create_jsonrpc_routes(
            request_handler=request_handler,
            rpc_url="/a2a/jsonrpc",
            enable_v0_3_compat=True,
        ),
        rest_routes=create_rest_routes(
            request_handler=request_handler,
            path_prefix="/a2a/rest",
            enable_v0_3_compat=True,
        ),
    )
    return app


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--host", default="127.0.0.1")
    parser.add_argument("--port", type=int, default=41241)
    args = parser.parse_args()
    logging.basicConfig(level=logging.INFO)
    uvicorn.run(create_app(args.host, args.port), host=args.host, port=args.port)


if __name__ == "__main__":
    main()

Overwriting c:\Users\tamtt.OFFICEVNPAY\Desktop\A2A\my_a2a_system/orchestrator_agent.py


In [21]:
# Khởi động orchestrator (cổng 41241) và chờ sẵn sàng
start_server("orchestrator_agent", 41241)
await wait_for_server("http://127.0.0.1:41241")
print("Orchestrator SẴN SÀNG ✅")

Đã khởi động orchestrator_agent (pid=33116) -> log: orchestrator_agent.log
Orchestrator SẴN SÀNG ✅


# Chương 9 — Demo End-to-End: gọi orchestrator 🎬

Bây giờ mọi thứ đã nối mạch: **User → Orchestrator → (A2A) → 3 Workers**.

Hãy chú ý:
- Dòng `[status]` in ra là **orchestrator** báo tiến trình.
- Câu trả lời cuối là **kết quả tổng hợp** — orchestrator đã âm thầm gọi các worker ở tầng dưới.


In [22]:
# Demo 1: câu hỏi đơn giản -> orchestrator gọi đúng 1 chuyên gia
ket_qua = await call_agent("http://127.0.0.1:41241", "bạn có những khả năng gì?")
print()
print("=== TRẢ LỜI CỦA ORCHESTRATOR ===")
print(ket_qua)

      [status] TASK_STATE_WORKING Đang điều phối các chuyên gia...
      [status] TASK_STATE_COMPLETED

=== TRẢ LỜI CỦA ORCHESTRATOR ===
Xin chào! Mình là trợ lý điều phối, có thể giúp bạn với nhiều loại yêu cầu khác nhau. Dưới đây là những khả năng của mình:

🌤️ **Thời tiết**
- Xem dự báo thời tiết, nhiệt độ, tình trạng mưa nắng tại các khu vực.

📰 **Tin tức**
- Tra cứu tin tức mới nhất, cập nhật tình hình thời sự trong và ngoài nước.

💱 **Tiền tệ / Đổi tiền**
- Chuyển đổi giữa các loại tiền tệ, xem tỷ giá quy đổi.

✈️ **Đặt vé máy bay**
- Hỗ trợ đặt vé máy bay theo yêu cầu của bạn.

🔀 **Yêu cầu phức tạp**
- Kết hợp nhiều thông tin cùng lúc (ví dụ: vừa xem thời tiết vừa đặt vé máy bay để bạn có kế hoạch đi lại phù hợp).

Bạn cần mình hỗ trợ gì hôm nay? 😊


In [23]:
# Demo 2: câu hỏi TỔNG HỢP -> orchestrator gọi NHIỀU chuyên gia
ket_qua = await call_agent(
    "http://127.0.0.1:41241",
    "Hôm nay thời tiết Hà Nội thế nào? Và 100 USD đổi được bao nhiêu VND?"
)
print()
print("=== TRẢ LỜI CỦA ORCHESTRATOR ===")
print(ket_qua)

      [status] TASK_STATE_WORKING Đang điều phối các chuyên gia...
      [status] TASK_STATE_COMPLETED

=== TRẢ LỜI CỦA ORCHESTRATOR ===
Tôi đã có kết quả cho cả hai câu hỏi của bạn:

**☀️ Thời tiết Hà Nội hôm nay:**
- Nhiệt độ khoảng **27°C**
- Trời trong xanh, khá đẹp
- Rất thích hợp để ra ngoài hoạt động
- Lưu ý nhớ mang theo nước uống nhé!

**💵 Đổi tiền tệ:**
- **100 USD = 2.450.000 VND**
- Tỷ giá: 1 USD = 24.500 VND

Hy vọng thông tin hữu ích với bạn! Nếu cần tra cứu thêm điều gì, cứ cho tôi biết nhé! 😊


In [24]:
# Demo 3: xem "danh thiếp" (Agent Card) của orchestrator - agent khai báo mình là ai
import httpx

async with httpx.AsyncClient() as client:
    r = await client.get("http://127.0.0.1:41241/.well-known/agent-card.json")
    card = r.json()

print("Tên:", card.get("name"))
print("Mô tả:", card.get("description"))
print("Kỹ năng:", [s["name"] for s in card.get("skills", [])])
print("Địa chỉ liên lạc:", [i["url"] for i in card.get("supported_interfaces", [])])

Tên: Orchestrator Agent
Mô tả: Điều phối viên: nghe yêu cầu, gọi các chuyên gia (weather/news/currency/booking) và tổng hợp.
Kỹ năng: ['Orchestrate specialists']
Địa chỉ liên lạc: []


# Chương 10 — Human-in-the-Loop (HITL): để người dùng xác nhận trước khi hành động 🧑⚖️

## 10.1 Vấn đề: agent tự ý "làm" mà không hỏi

Hãy tưởng tượng agent **Booking** (đặt vé máy bay) nhận được yêu cầu:

> "Đặt cho tôi một vé Hà Nội → Đà Nẵng."

Nếu agent lập tức đặt vé và trừ tiền ngay — có thể bạn sẽ ngỡ ngàng: "Ơ, tôi chưa đồng ý mà?" 😱

Với những hành động **rủi ro / tốn tiền** (đặt vé, thanh toán, xoá dữ liệu...), agent phải
**dừng lại và hỏi người dùng xác nhận** trước khi làm. Cơ chế đó gọi là
**Human-in-the-Loop (HITL)** — "con người ở trong vòng lặp".

## 10.2 A2A làm HITL bằng trạng thái `input-required`

A2A hỗ trợ sẵn HITL qua **một trạng thái Task đặc biệt** trong vòng đời:

| Trạng thái | Ý nghĩa |
|---|---|
| `submitted` → `working` | Đã nhận, đang xử lý |
| **`input-required`** | ⏸️ **Agent cần người dùng trả lời mới tiếp tục được** |
| `completed` | Xong (sau khi người dùng đã trả lời) |

Quy trình HITL trong A2A có **3 bước**:

```mermaid
sequenceDiagram
    participant C as Client (Notebook)
    participant B as Booking Agent
    C->>B: SendMessage("Đặt vé Hà Nội -> Đà Nẵng")
    B-->>C: status: working
    B-->>C: status: INPUT-REQUIRED "Bạn xác nhận đặt vé...? (có/không)"
    Note over C: ⏸️ Hỏi người dùng
    C->>B: SendMessage(CÙNG task_id, "có")
    B-->>C: status: working
    B-->>C: artifact: "ĐẶT VÉ THÀNH CÔNG!"
    B-->>C: status: completed
```

> 🔑 **Điểm mấu chốt:** Khi agent cần input, nó **KHÔNG hoàn tất** task. Nó gửi trạng thái
> `input-required` kèm câu hỏi rồi **"nhường quyền"**. Client phải gửi câu trả lời lại cho
> **CÙNG task** (cùng `task_id` và `context_id`) thì agent mới "tỉnh dậy" và tiếp tục.

## 10.3 Ta sẽ xây gì?

Ta thêm **Booking Agent** (worker thứ 4, cổng **41254**) và nối nó vào **orchestrator**:

```mermaid
graph TD
    U["👤 Người dùng"] -->|A2A| O["🧑‍💼 Orchestrator"]
    O -->|A2A| B["🎫 Booking Agent<br>port 41254 - HITL"]
    O -->|A2A| W["🌤️ Weather"]
    O -->|A2A| N["📰 News"]
    O -->|A2A| C["💱 Currency"]
    B -.->|input-required: cần xác nhận| U
```

Có **3 khối xây dựng**:

1. **Booking Agent** (worker) — biết cách "hỏi" người dùng bằng `input-required`.
2. **`call_agent` nâng cấp** (client) — nhận biết `input-required`, hỏi người dùng rồi gửi câu trả lời lại.
3. **Orchestrator nâng cấp** — chuyển tiếp câu hỏi HITL từ worker lên người dùng.

## 10.4 Khối 1 — Booking Agent: worker biết cách "hỏi" 🎫 (dùng LLM thật)

`booking_agent.py` giống hệt các worker trước ở **khung A2A** (Agent Card, routes...),
và giờ cũng có **bộ não DeepAgent THẬT** (DeepSeek) như weather/news/currency:

- `build_deep_agent()` tạo DeepAgent với tool **`prepare_booking(from_city, to_city)`** (tra giá vé).
- Khi nhận yêu cầu, DeepAgent **dùng LLM để hiểu** yêu cầu (trích nơi đi / nơi đến), gọi tool
  lấy giá, rồi **PHÂN LOẠI**: chỉ **HITL khi thật sự cần xác nhận** (hành động đặt vé/tốn tiền),
  còn **hỏi thông tin** (giá vé, chặng bay, lịch bay...) thì **trả lời trực tiếp, không HITL**.
- Cách DeepAgent "báo hiệu" cần xác nhận: trả về câu trả lời **bắt đầu bằng tiền tố**
  `__NEED_CONFIRM__::`. Executor nhìn tiền tố này để quyết định chuyển task sang `input-required`
  (HITL) hay `complete()` luôn — đúng như bạn muốn: **HITL chỉ kích hoạt khi agent cần báo
  lên (orchestrator) rằng phải có xác nhận của người dùng**, không phải lúc nào cũng hỏi ở lượt đầu.
- DeepAgent được tạo với **`checkpointer=InMemorySaver()`** → agent **CÓ TRÍ NHỚ** theo
  `thread_id` (= `task_id`). Nhờ đó trong vòng lặp hỏi lại, agent vẫn nhớ yêu cầu gốc
  và các lượt trả lời trước để soạn câu hỏi tiếp theo đúng ngữ cảnh.

Phần **HITL** (vòng đời task) vẫn do `BookingExecutor` xử lý, giờ là **LINH HOẠT + VÒNG LẶP**:

- **Lần gọi 1** (task mới): chạy DeepAgent rồi **kiểm tra tiền tố**:
  - Không có `__NEED_CONFIRM__::` → yêu cầu là **hỏi thông tin** → trả lời bình thường,
    gọi `complete()` (KHÔNG HITL).
  - Có `__NEED_CONFIRM__::` → yêu cầu là **đặt vé** → bỏ tiền tố, lấy câu hỏi xác nhận →
    gọi `updater.requires_input(message=câu hỏi)` để chuyển task sang `input-required`,
    rồi **trả về** (KHÔNG gọi `complete()`).
- **Lần gọi 2+** (resume, cùng `task_id`): framework gọi lại `execute()`; đọc câu trả lời từ
  `context.get_user_input()` → **dùng LLM hiểu ý** người dùng:
  - ✅ ĐỒNG Ý → `complete()` (đặt vé thành công).
  - ❌ **CHƯA đồng ý** → chạy DeepAgent (nhờ checkpointer nhớ hội thoại) để **soạn câu hỏi
    tiếp theo**, gọi `requires_input(...)` rồi **trả về** → framework lại gọi `execute()`
    khi người dùng trả lời → **LẶP LẠI** cho tới khi xác nhận thật sự.

```mermaid
sequenceDiagram
    participant N as Notebook (call_agent + ask_user)
    participant B as 🎫 Booking Agent
    alt Hỏi thông tin (giá vé, chặng bay...)
        N->>B: SendMessage("Giá vé Hà Nội - Đà Nẵng?")
        B-->>N: artifact trả lời + completed (KHÔNG HITL)
    else Đặt vé (tốn tiền)
        N->>B: SendMessage("Đặt vé Hà Nội -> Đà Nẵng")
        B-->>N: status: input-required "<câu hỏi xác nhận>"
        loop cho tới khi xác nhận
            N->>N: 🧑‍⚖️ hỏi người dùng
            N->>B: SendMessage(task_id, câu trả lời)
            B->>B: LLM phân loại: đồng ý?
            alt ❌ Chưa đồng ý
                B->>B: DeepAgent (nhớ hội thoại) soạn câu hỏi mới
                B-->>N: status: input-required "câu hỏi mới"
                N->>N: 🧑‍⚖️ hỏi lại người dùng
            else ✅ Đồng ý
                B-->>N: artifact "ĐẶT VÉ THÀNH CÔNG!" + completed
            end
        end
    end
```

### 🧠 Nhận diện câu trả lời KHÔNG hardcode — dùng LLM phân loại

Ở mỗi lần gọi resume, nếu ta liệt kê cứng `if answer in ("có", "ok", "yes"...):` thì người dùng gõ
**"okie"**, **"chốt"**, **"chắc chắn rồi"**, **"yes please"**... sẽ KHÔNG được coi là đồng ý
→ bị hiểu nhầm thành huỷ vé. 😅

Giải pháp: `BookingExecutor._is_confirmed(answer)` **hỏi thẳng LLM**:

> "Người dùng trả lời: 'okie'. Họ có ĐỒNG Ý đặt vé không? Chỉ trả lời: CÓ hoặc KHÔNG."

LLM hiểu ngữ cảnh tiếng Việt/tiếng lóng nên bắt được mọi cách diễn đạt, không cần liệt kê từ khoá.
Nếu không có API key, phương thức `_keyword_is_confirmed()` sẽ dùng danh sách từ khoá làm fallback.

> 💡 **Phân chia trách nhiệm:** LLM lo 2 việc "trí thông minh": (1) **phân loại yêu cầu
> (hỏi thông tin vs đặt vé) & soạn câu hỏi**, (2) **hiểu câu trả lời** của người dùng.
> Phần quyết định đặt/huỷ vẫn là logic hệ thống (state machine) — minh bạch, kiểm soát được,
> đúng triết lý HITL.
>
> 🛡️ **Phòng thủ:** nếu LLM trả về trống sau tiền tố → dùng câu hỏi mặc định;
> nếu LLM trả lời phân loại không rõ ràng → mặc định **KHÔNG xác nhận** (an toàn, không tính phí)
> → agent tiếp tục hỏi lại (vòng lặp), không tự ý đặt vé.

Chạy cell dưới để ghi file `booking_agent.py`.

In [25]:
%%writefile {PROJECT_DIR}/booking_agent.py
"""Booking Agent - một A2A Server minh hoạ HUMAN-IN-THE-LOOP (HITL), dùng LLM thật.

Booking Agent có "bộ não" là DeepAgent THẬT (DeepSeek) để:
  - Hiểu yêu cầu đặt vé bằng ngôn ngữ tự nhiên.
  - Gọi tool prepare_booking để lấy giá vé.
  - PHÂN LOẠI yêu cầu: chỉ HITL khi THẬT SỰ cần xác nhận (hành động tốn tiền),
    còn hỏi thông tin (giá vé, chặng bay...) thì trả lời trực tiếp.
  - SOẠN câu hỏi xác nhận cho người dùng.

LINH HOẠT - HITL CHỈ KHI CẦN (không phải lúc nào cũng hỏi ở lượt đầu):
  0. Lần gọi 1: DeepAgent phân loại yêu cầu:
     - Chỉ HỎI THÔNG TIN (giá vé, chặng bay, lịch bay...) -> trả lời bình thường
       + complete(), KHÔNG cần người dùng xác nhận (không HITL).
     - ĐẶT VÉ (hành động tốn tiền) -> trả về "<CONFIRM_PREFIX><câu hỏi xác nhận>"
       -> publish `requires_input(...)` rồi TRẢ VỀ (nhường quyền cho người dùng).
  1. Lần gọi 2+ (resume, cùng task_id): đọc câu trả lời.
     - Xác nhận -> complete() (đặt vé).
     - Chưa xác nhận -> chạy DeepAgent (với checkpointer nhớ cả hội thoại) để
       soạn câu hỏi TIẾP THEO, gọi requires_input(...) và TRẢ VỀ -> LẶP LẠI.

Câu trả lời của người dùng cũng được hiểu BẰNG LLM (KHÔNG hardcode từ khoá):
  - 'ok', 'okie', 'chốt', 'chắc chắn rồi', 'yes please'... -> ĐỒNG Ý -> đặt vé.
  - Các câu trả lời khác -> agent HỎI LẠI (vòng lặp) cho tới khi người dùng xác nhận thật sự.
  Nếu chưa có API key, dùng fallback nhận diện từ khoá (mock).
"""
import argparse
import logging

import uvicorn
from fastapi import FastAPI
from langchain_core.messages import HumanMessage

from a2a.server.agent_execution.agent_executor import AgentExecutor
from a2a.server.agent_execution.context import RequestContext
from a2a.server.events.event_queue import EventQueue
from a2a.server.request_handlers import DefaultRequestHandler
from a2a.server.routes import (
    add_a2a_routes_to_fastapi,
    create_agent_card_routes,
    create_jsonrpc_routes,
    create_rest_routes,
)
from a2a.server.tasks.inmemory_task_store import InMemoryTaskStore
from a2a.server.tasks.task_updater import TaskUpdater
from a2a.types import (
    AgentCapabilities,
    AgentCard,
    AgentInterface,
    AgentProvider,
    AgentSkill,
    Part,
    Task,
    TaskState,
    TaskStatus,
)
from langgraph.checkpoint.memory import InMemorySaver
from a2a_common import get_model, run_deep_agent

logger = logging.getLogger(__name__)

# Tiền tố đánh dấu "yêu cầu này cần người dùng xác nhận (HITL)".
# LLM chỉ thêm tiền tố này khi yêu cầu là HÀNH ĐỘNG tốn tiền (đặt vé);
# khi chỉ hỏi thông tin (giá vé, chặng bay...) thì KHÔNG thêm -> trả lời luôn.
CONFIRM_PREFIX = "__NEED_CONFIRM__::"

# Câu hỏi xác nhận mặc định (phòng thủ khi LLM trả về trống sau tiền tố)
DEFAULT_QUESTION = (
    "Bạn xác nhận đặt vé máy bay? (trả lời 'có' hoặc 'không')"
)

# Giá vé giả lập (VND) cho từng cặp chặng bay
FLIGHT_PRICES = {
    ("hanoi", "danang"): 1_500_000,
    ("danang", "hanoi"): 1_500_000,
    ("hanoi", "hcmc"): 2_200_000,
    ("hcmc", "hanoi"): 2_200_000,
}

# Tên gọi tắt của các thành phố (để chuẩn hoá từ ngôn ngữ tự nhiên -> key nội bộ)
CITY_ALIASES = {
    "hanoi": ["hà nội", "hanoi", "ha noi", "hn"],
    "danang": ["đà nẵng", "danang", "da nang", "dn"],
    "hcmc": ["hồ chí minh", "hcmc", "sài gòn", "saigon", "hcm"],
}


def normalize_city(name: str) -> str:
    """Chuẩn hoá tên thành phố do LLM truyền vào -> key nội bộ (hanoi/danang/hcmc)."""
    n = name.strip().lower()
    for key, aliases in CITY_ALIASES.items():
        if any(a in n for a in aliases):
            return key
    return n  # không nhận diện được -> giữ nguyên (sẽ dùng giá fallback)


def build_deep_agent(model=None):
    """Tạo DeepAgent làm "bộ não" thật. Chưa có API key -> None (dùng mock)."""
    if model is None:
        model = get_model()
    if model is None:
        return None

    from deepagents import create_deep_agent
    from langchain.tools import tool

    @tool
    def prepare_booking(from_city: str, to_city: str) -> str:
        """Chuẩn bị vé: trả về giá vé (VND) cho một chặng bay."""
        price = FLIGHT_PRICES.get(
            (normalize_city(from_city), normalize_city(to_city)), 1_200_000
        )
        return f"Giá vé {from_city} → {to_city} là {price:,} VND."

    # checkpointer=InMemorySaver() -> agent CÓ TRÍ NHỚ theo thread_id (= task_id),
    # nhờ đó vòng lặp HITL hỏi lại vẫn nhớ yêu cầu gốc và các lượt trước.
    return create_deep_agent(
        name="booking_agent",
        model=model,
        tools=[prepare_booking],
        system_prompt=(
            "Bạn là chuyên gia đặt vé máy bay.\n"
            "PHÂN LOẠI yêu cầu của người dùng trước khi trả lời:\n"
            "1. Nếu chỉ HỎI THÔNG TIN (giá vé, chặng bay, lịch bay, so sánh giá...) -> "
            "trả lời trực tiếp bằng tiếng Việt, KHÔNG cần xác nhận, "
            f"KHÔNG thêm tiền tố '{CONFIRM_PREFIX}'.\n"
            "2. Nếu yêu cầu ĐẶT VÉ (đặt/book/order/mua vé — hành động tốn tiền) -> "
            "trích 'nơi đi' và 'nơi đến', gọi tool prepare_booking để lấy giá vé, "
            "rồi trả lời ĐÚNG MỘT câu hỏi xác nhận ngắn gọn BẮT ĐẦU BẰNG "
            f"'{CONFIRM_PREFIX}', ví dụ:\n"
            f"   '{CONFIRM_PREFIX}Bạn xác nhận đặt vé Hà Nội → Đà Nẵng giá 1,500,000 VND? (có/không)'\n"
            f"CHỈ thêm tiền tố '{CONFIRM_PREFIX}' khi THẬT SỰ cần người dùng xác nhận "
            "(đặt vé = tốn tiền).\n"
            "KHÔNG tự ý xác nhận vé. Nếu người dùng chưa đồng ý (trả lời khác 'có'), "
            "hãy HỎI LẠI theo ngữ cảnh, KHÔNG kết thúc đặt vé."
        ),
        checkpointer=InMemorySaver()
    )


def parse_booking(query: str):
    """(Chỉ dùng cho mock brain) Trích "nơi đi" và "nơi đến" từ câu hỏi."""
    q = query.lower()
    cities = {
        "hà nội": "hanoi", "hanoi": "hanoi", "ha noi": "hanoi",
        "đà nẵng": "danang", "danang": "danang", "da nang": "danang",
        "hồ chí minh": "hcmc", "hcmc": "hcmc", "sài gòn": "hcmc",
    }
    found = [name for key, name in cities.items() if key in q]
    if len(found) >= 2:
        return found[0], found[1]
    if len(found) == 1:
        return found[0], "danang"
    return None, None


class BookingExecutor(AgentExecutor):
    """Bộ não xử lý. `pending` nhớ các task đang chờ người dùng xác nhận."""

    def __init__(self, agent, llm=None):
        self.agent = agent   # DeepAgent: phân loại & soạn câu hỏi xác nhận (có checkpointer -> nhớ hội thoại)
        self.llm = llm       # LLM thường: hiểu câu trả lời của người dùng
        self.pending = {}    # task_id -> True (đang chờ xác nhận)

    async def execute(self, context: RequestContext, event_queue: EventQueue):
        task_id = context.task_id
        context_id = context.context_id
        user_input = context.get_user_input()
        updater = TaskUpdater(event_queue, task_id, context_id)

        # ============ LẦN GỌI ĐẦU: LLM quyết định có cần xác nhận hay không ============
        if task_id not in self.pending:
            # BẮT BUỘC: gửi Task (submitted) TRƯỚC các status update
            await event_queue.enqueue_event(
                Task(
                    id=task_id,
                    context_id=context_id,
                    status=TaskStatus(state=TaskState.TASK_STATE_SUBMITTED),
                    history=[context.message],
                )
            )

            await updater.start_work(
                message=updater.new_agent_message(
                    parts=[Part(text="Đang xử lý yêu cầu...")]
                )
            )

            # DeepAgent phân loại yêu cầu:
            #  - Hỏi thông tin (giá vé, chặng bay...) -> trả lời bình thường, KHÔNG HITL.
            #  - Đặt vé (hành động tốn tiền) -> trả về "<CONFIRM_PREFIX><câu hỏi>".
            answer = (await run_deep_agent(
                self.agent, user_input, thread_id=task_id
            )).strip()

            # Trường hợp 1: KHÔNG cần xác nhận -> trả lời bình thường, complete().
            if not answer.startswith(CONFIRM_PREFIX):
                await updater.add_artifact(
                    parts=[Part(text=answer)],
                    name="response",
                    last_chunk=True,
                )
                await updater.complete()
                return

            # Trường hợp 2: CẦN xác nhận -> bỏ tiền tố, lấy câu hỏi, sang HITL.
            question = answer[len(CONFIRM_PREFIX):].strip()
            if not question:  # phòng thủ: LLM trả về trống sau tiền tố
                question = DEFAULT_QUESTION

            self.pending[task_id] = True
            # HITL: chuyển task sang input-required và TRẢ VỀ (nhường quyền).
            # KHÔNG gọi complete() - chờ người dùng trả lời ở lần gọi sau.
            await updater.requires_input(
                message=updater.new_agent_message(parts=[Part(text=question)])
            )
            return

        # ============ LẦN GỌI THỨ 2+: vòng lặp xác nhận ============
        self.pending.pop(task_id)
        # Hiểu ý người dùng BẰNG LLM (flexible: 'okie', 'chốt'... đều bắt được).
        confirmed = await self._is_confirmed(user_input)

        if confirmed:
            await updater.start_work(
                message=updater.new_agent_message(
                    parts=[Part(text="Đang xác nhận đặt vé...")]
                )
            )
            await updater.add_artifact(
                parts=[Part(
                    text=(
                        "✅ ĐẶT VÉ THÀNH CÔNG!\n"
                        f"- Mã vé: BK-{task_id[:8].upper()}\n"
                        "Cảm ơn bạn đã sử dụng dịch vụ đặt vé!"
                    )
                )],
                name="response",
                last_chunk=True,
            )
            await updater.complete()
        else:
            # CHƯA xác nhận -> HỎI LẠI (vòng lặp HITL), KHÔNG complete().
            # Nhờ checkpointer (InMemorySaver), agent nhớ cả cuộc hội thoại theo
            # thread_id=task_id nên câu hỏi tiếp theo luôn đúng ngữ cảnh.
            if self.agent is not None:
                reply = await run_deep_agent(
                    self.agent, user_input, thread_id=task_id
                )
            else:
                reply = (
                    "Bạn vẫn muốn đặt vé chứ? Nếu có, hãy trả lời 'có' để tôi đặt. "
                    "Nếu muốn đổi chặng, hãy nói rõ nơi đi và nơi đến nhé!"
                )
            reply = reply.strip()
            # Đã chắc chắn "chưa xác nhận" -> luôn cần hỏi lại, nên bỏ tiền tố
            # (nếu LLM vẫn thêm) và dùng phần còn lại làm câu hỏi.
            if reply.startswith(CONFIRM_PREFIX):
                reply = reply[len(CONFIRM_PREFIX):].strip()
            if not reply:
                reply = (
                    "Bạn vẫn muốn đặt vé chứ? Nếu có, hãy trả lời 'có' để tôi đặt nhé!"
                )

            self.pending[task_id] = True  # vẫn đang chờ xác nhận (vòng lặp)
            await updater.requires_input(
                message=updater.new_agent_message(parts=[Part(text=reply)])
            )
            return

    async def _is_confirmed(self, answer: str) -> bool:
        """Hiểu câu trả lời của người dùng có phải "đồng ý" không.

        - Có LLM: hỏi LLM phân loại (hiểu theo ngữ cảnh, KHÔNG cần liệt kê từ khoá).
        - Không có LLM: fallback nhận diện từ khoá (mock).
        """
        if self.llm is not None:
            prompt = (
                "Người dùng vừa trả lời câu hỏi xác nhận đặt vé máy bay.\n"
                f"Câu trả lời của họ: '{answer}'\n"
                "Họ có ĐỒNG Ý đặt vé không?\n"
                "Chỉ trả lời đúng MỘT từ: CÓ hoặc KHÔNG."
            )
            result = await self.llm.ainvoke([HumanMessage(content=prompt)])
            return self._parse_yes_no(str(result.content))
        return self._keyword_is_confirmed(answer)

    @staticmethod
    def _parse_yes_no(reply: str) -> bool:
        """Đọc 'bản án' CÓ/KHÔNG do LLM trả về."""
        upper = reply.strip().upper()
        if "KHÔNG" in upper or upper in ("NO", "N"):
            return False
        if "CÓ" in upper or upper in ("OK", "OKE", "OKIE", "YES", "YEAH", "YEP"):
            return True
        return False  # không rõ -> mặc định KHÔNG xác nhận (an toàn, không tính phí)

    @staticmethod
    def _keyword_is_confirmed(answer: str) -> bool:
        """Fallback (không có LLM): nhận diện bằng từ khoá."""
        a = answer.strip().lower()
        yes_words = [
            "có", "ok", "oke", "okie", "yes", "yeah", "yep",
            "đồng ý", "xác nhận", "chốt", "chuẩn", "đúng", "chắc chắn",
            "agree", "confirmed", "go ahead",
        ]
        return any(w in a for w in yes_words)


    async def cancel(self, context: RequestContext, event_queue: EventQueue):
        updater = TaskUpdater(event_queue, context.task_id, context.context_id)
        await updater.cancel()


def create_app(host: str, port: int):
    """Lắp ráp server: Agent Card + Request Handler + routes."""
    # Lấy model MỘT lần: dùng cho cả soạn câu hỏi (DeepAgent) và hiểu câu trả lời (LLM)
    llm = get_model()
    agent = build_deep_agent(llm)

    agent_card = AgentCard(
        name="Booking Agent",
        description="Chuyên gia đặt vé máy bay (LLM): hỏi người dùng xác nhận (HITL) chỉ khi cần đặt vé tốn tiền.",
        provider=AgentProvider(organization="A2A Course", url="http://example.com"),
        version="1.0.0",
        capabilities=AgentCapabilities(streaming=True, push_notifications=False),
        default_input_modes=["text"],
        default_output_modes=["text", "task-status"],
        skills=[
            AgentSkill(
                id="booking",
                name="Flight booking with confirmation",
                description="Đặt vé máy bay; yêu cầu người dùng xác nhận (input-required) khi cần.",
                tags=["booking", "hitl"],
                examples=["Đặt vé Hà Nội đi Đà Nẵng"],
                input_modes=["text"],
                output_modes=["text", "task-status"],
            )
        ],
        supported_interfaces=[
            AgentInterface(
                protocol_binding="JSONRPC",
                protocol_version="1.0",
                url=f"http://{host}:{port}/a2a/jsonrpc",
            )
        ],
    )

    request_handler = DefaultRequestHandler(
        agent_executor=BookingExecutor(agent, llm),
        task_store=InMemoryTaskStore(),
        agent_card=agent_card,
    )

    app = FastAPI(title=agent_card.name)
    add_a2a_routes_to_fastapi(
        app,
        agent_card_routes=create_agent_card_routes(agent_card=agent_card),
        jsonrpc_routes=create_jsonrpc_routes(
            request_handler=request_handler,
            rpc_url="/a2a/jsonrpc",
            enable_v0_3_compat=True,
        ),
        rest_routes=create_rest_routes(
            request_handler=request_handler,
            path_prefix="/a2a/rest",
            enable_v0_3_compat=True,
        ),
    )
    return app


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--host", default="127.0.0.1")
    parser.add_argument("--port", type=int, default=41254)
    args = parser.parse_args()
    logging.basicConfig(level=logging.INFO)
    uvicorn.run(create_app(args.host, args.port), host=args.host, port=args.port)


if __name__ == "__main__":
    main()

Overwriting c:\Users\tamtt.OFFICEVNPAY\Desktop\A2A\my_a2a_system/booking_agent.py


In [26]:
# Khởi động lại toàn bộ để nạp file mới (booking_agent + orchestrator đã nâng cấp)
stop_all_servers()
start_server("weather_agent", 41251)
start_server("news_agent", 41252)
start_server("currency_agent", 41253)
start_server("booking_agent", 41254)

# Chờ cả 4 worker sẵn sàng
for name, port in [("weather", 41251), ("news", 41252), ("currency", 41253), ("booking", 41254)]:
    url = f"http://127.0.0.1:{port}"
    ok = await wait_for_server(url)
    print(f"{name:10s} {url:35s} -> {'SẴN SÀNG ✅' if ok else 'LỖI ❌'}")

# Khởi động orchestrator (bản đã thêm ask_booking + HITL relay)
start_server("orchestrator_agent", 41241)
await wait_for_server("http://127.0.0.1:41241")
print("Orchestrator SẴN SÀNG ✅")

Đã dừng tất cả server.
Đã khởi động weather_agent (pid=29376) -> log: weather_agent.log
Đã khởi động news_agent (pid=36564) -> log: news_agent.log
Đã khởi động currency_agent (pid=12276) -> log: currency_agent.log
Đã khởi động booking_agent (pid=16544) -> log: booking_agent.log
weather    http://127.0.0.1:41251              -> SẴN SÀNG ✅
news       http://127.0.0.1:41252              -> SẴN SÀNG ✅
currency   http://127.0.0.1:41253              -> SẴN SÀNG ✅
booking    http://127.0.0.1:41254              -> SẴN SÀNG ✅
Đã khởi động orchestrator_agent (pid=39804) -> log: orchestrator_agent.log
Orchestrator SẴN SÀNG ✅


### ▶️ Demo A — gọi Booking Agent trực tiếp

Chạy cell dưới và trả lời câu hỏi xác nhận. Nhờ LLM phân loại, bạn có thể trả lời
**bất kỳ cách nào**: `có`, `okie`, `chốt`, `chắc chắn rồi`, `yes please`... đều được coi là
**đồng ý** (đặt vé). Trả lời `không`, `thôi`, `huỷ`... thì agent huỷ vé.

In [ ]:
# Hàm "hỏi người dùng" tương tác: in câu hỏi rồi đọc câu trả lời từ bàn phím
def ask_user_interactive(question):
    print(f"\n🧑‍⚖️  CẦN NGƯỜI DÙNG XÁC NHẬN: {question}")
    return input("✍️  Câu trả lời của bạn: ")


# Demo A: gọi TRỰC TIẾP Booking Agent (1 chặng HITL)
ket_qua = await call_agent(
    "http://127.0.0.1:41254",
    "Đặt cho tôi một vé máy bay từ Hà Nội đến Đà Nẵng",
    ask_user=ask_user_interactive,
)
print()
print("=== KẾT QUẢ BOOKING ===")
print(ket_qua)

      [status] TASK_STATE_WORKING Đang xử lý yêu cầu...
      [status] TASK_STATE_INPUT_REQUIRED Bạn xác nhận đặt vé Hà Nội → Đà Nẵng giá 1,500,000 VND? (có/không)

🧑‍⚖️  CẦN NGƯỜI DÙNG XÁC NHẬN: Bạn xác nhận đặt vé Hà Nội → Đà Nẵng giá 1,500,000 VND? (có/không)
      [status] TASK_STATE_INPUT_REQUIRED Bạn đang hỏi về thông tin các chuyến bay Hà Nội → Đà Nẵng. Tuy nhiên, tôi chỉ có thể cung cấp giá vé cho chặng bay này là **1,500,000 VND**.

Hiện tại, tôi không có thông tin chi tiết về danh sách các chuyến bay cụ thể (giờ bay, hãng hàng không, số hiệu chuyến bay...) cho chặng này.

Bạn có muốn tôi tiếp tục xác nhận đặt vé Hà Nội → Đà Nẵng với giá **1,500,000 VND** không? (có/không)

🧑‍⚖️  CẦN NGƯỜI DÙNG XÁC NHẬN: Bạn đang hỏi về thông tin các chuyến bay Hà Nội → Đà Nẵng. Tuy nhiên, tôi chỉ có thể cung cấp giá vé cho chặng bay này là **1,500,000 VND**.

Hiện tại, tôi không có thông tin chi tiết về danh sách các chuyến bay cụ thể (giờ bay, hãng hàng không, số hiệu chuyến bay...) cho chặn

## 10.6 Khối 3 — Orchestrator nâng cấp: "chuyển tiếp" HITL dạng VÒNG LẶP 🧑‍💼

`orchestrator_agent.py` (viết ở Chương 8) được nâng cấp thêm:

- Tool **`ask_booking`** → gọi Booking Agent (cổng 41254).
- Hệ thống prompt: hướng dẫn DeepAgent **trả NGUYÊN VĂN** chuỗi `__HITL__::...` khi gặp.
- `OrchestratorExecutor` xử lý **nhiều lượt chạy (vòng lặp)**:
  - **Lượt 1:** chạy DeepAgent → nếu câu trả lời bắt đầu bằng `__HITL__::`, trích câu hỏi,
    gọi `updater.requires_input(...)` (chuyển task của CHÍNH orchestrator sang `input-required`),
    nhớ thông tin worker trong `self.pending`, rồi **trả về** (nhường quyền).
  - **Lượt 2+ (resume):** người dùng đã trả lời → gọi
    `call_agent(worker_url, answer, task_id=..., context_id=...)` để chuyển xuống **CÙNG task**
    của worker. Sau đó orchestrator **kiểm tra kết quả**:
    - Nếu worker **lại trả về** `__HITL__::...` (nghĩa là worker vẫn cần hỏi thêm) →
      orchestrator **trích câu hỏi mới**, cập nhật `self.pending` với `task_id`/`context_id`
      mới của worker, gọi `requires_input(...)` rồi **trả về** → **LẶP LẠI** lượt chuyển tiếp.
    - Nếu worker đã **hoàn tất** (không còn `__HITL__::`) → orchestrator trả kết quả cuối.

Toàn bộ chuỗi HITL qua **2 tầng, lặp tới khi xác nhận**:

```mermaid
sequenceDiagram
    participant N as Notebook (call_agent + ask_user)
    participant O as 🧑‍💼 Orchestrator
    participant B as 🎫 Booking Agent
    N->>O: "Đặt vé Hà Nội -> Đà Nẵng"
    O->>O: DeepAgent chọn ask_booking
    O->>B: call_agent(booking)
    B-->>O: __HITL__::<task_id>|<ctx_id>|<câu hỏi>
    O-->>N: status: input-required "<câu hỏi>"
    loop cho tới khi worker hoàn tất
        N->>N: 🧑‍⚖️ hỏi người dùng
        N->>O: SendMessage(CÙNG task_id, câu trả lời)
        O->>B: call_agent(task_id cũ, câu trả lời)
        alt B vẫn cần hỏi (chưa xác nhận)
            B-->>O: __HITL__::<task_id mới>|<ctx_id>|<câu hỏi mới>
            O-->>N: input-required "<câu hỏi mới>" (tiếp tục vòng lặp)
        else B hoàn tất
            B-->>O: "ĐẶT VÉ THÀNH CÔNG!"
            O-->>N: artifact + completed
        end
    end
```

> 🧩 **Lưu ý "tai nạn" của DeepAgents:** nếu tool `ask_booking` **ném ngoại lệ** để báo HITL,
> DeepAgents sẽ **nuốt** ngoại lệ đó (coi như tool lỗi) — ngoại lệ không lan lên executor.
> Vì vậy ta dùng cách **trả về chuỗi đánh dấu** `__HITL__::...` thay vì ném exception, rồi
> executor "bắt" chuỗi đó sau khi DeepAgent chạy xong — và **kiểm tra lại** sau mỗi lượt
> chuyển tiếp để duy trì vòng lặp.

In [ ]:
# Hàm "hỏi người dùng" (định nghĩa lại cho an toàn nếu chạy cell này riêng lẻ)
def ask_user_interactive(question):
    print(f"\n🧑‍⚖️  CẦN NGƯỜI DÙNG XÁC NHẬN: {question}")
    return input("✍️  Câu trả lời của bạn: ")


# Demo B: đặt vé qua ORCHESTRATOR -> orchestrator gọi Booking Agent -> agent hỏi xác nhận
ket_qua = await call_agent(
    "http://127.0.0.1:41241",
    "Đặt giúp tôi một vé máy bay từ Hà Nội đi Đà Nẵng",
    ask_user=ask_user_interactive,
)
print()
print("=== TRẢ LỜI CỦA ORCHESTRATOR ===")
print(ket_qua)

## 10.7 Tổng kết HITL 🏁

| Thành phần | Vai trò |
|---|---|
| **Booking Agent (LLM)** | DeepAgent hiểu yêu cầu & soạn câu hỏi; `requires_input()` để tạm dừng; hỏi lại tới khi xác nhận thật sự (vòng lặp) |
| **`call_agent(ask_user=...)`** | Client nhận biết `input-required`, hỏi người dùng, gửi câu trả lời lại CÙNG task (lặp tự nhiên) |
| **Orchestrator** | Chuyển tiếp câu hỏi HITL lên người dùng và câu trả lời xuống worker; kiểm tra lại sau mỗi lượt để duy trì vòng lặp |

**Điểm chốt cần nhớ:**
- HITL trong A2A = trạng thái **`input-required`** trong vòng đời Task.
- Agent "hỏi" bằng cách: gửi `input-required` kèm câu hỏi, rồi **trả về** (không gọi `complete()`).
- **Vòng lặp:** chừng nào người dùng chưa xác nhận thật sự, agent lại `requires_input(...)` +
  trả về → framework gọi lại `execute()` khi có câu trả lời mới → lặp tới khi `complete()`.
- Client "trả lời" bằng cách: gửi tin nhắn mới với **cùng `task_id` / `context_id`**.
- LLM lo phần "trí thông minh": **hiểu yêu cầu & soạn câu hỏi** + **hiểu câu trả lời** (flexible —
  "okie", "chốt", "thôi"... đều bắt được); quyết định đặt/huỷ vẫn là logic hệ thống.
- **Checkpointer** (`InMemorySaver`) giúp agent **nhớ hội thoại** theo `thread_id` = `task_id`,
  nhờ vậy câu hỏi hỏi lại luôn đúng ngữ cảnh (không "quên" yêu cầu ban đầu).
- Orchestrator chỉ là **người chuyển tiếp**: worker hỏi → orchestrator hỏi hộ → người dùng trả lời
  → orchestrator chuyển xuống worker → worker lại hỏi → lặp... → worker hoàn tất → orchestrator
  trả kết quả cuối.

## Bài tập về nhà 🏠

1. Chạy Demo A và thử **nhiều lượt**: trả lời `để lúc khác` → agent hỏi lại → trả lời `thôi` →
   agent hỏi lại → cuối cùng trả lời `okie` → quan sát agent hiểu đúng ý và đặt vé.
2. Chạy Demo B (qua orchestrator) với cùng kịch bản nhiều lượt — quan sát orchestrator chuyển tiếp
   qua lại cho tới khi xác nhận thật sự.
3. Mở rộng Booking Agent: hỏi **nhiều bước** (xác nhận vé → xác nhận thanh toán) bằng cách dùng
   nhiều trạng thái `pending` (vd `pending` lưu `bước hiện tại`).
4. Thêm một agent HITL khác (vd `payment_agent.py` yêu cầu xác nhận thanh toán) rồi nối vào orchestrator.
5. Đọc `refs/A2A/docs/topics/life-of-a-task.md` để hiểu sâu hơn về `input-required`.

# Chương 11 — Tổng kết 🏁

## Sơ đồ toàn cảnh

```mermaid
graph TD
    U([👤 User]) -->|A2A| O
    subgraph O[🧑‍💼 Orchestrator - port 41241]
        DA[DeepAgent<br/>tools: ask_weather/news/currency/booking]
        EX[AgentExecutor]
        DA --> EX
    end
    O -->|A2A Client| W
    O -->|A2A Client| N
    O -->|A2A Client| C
    O -->|A2A Client| B
    W[🌤️ Weather - 41251<br/>DeepAgent + tool get_weather]
    N[📰 News - 41252<br/>DeepAgent + tool get_news]
    C[💱 Currency - 41253<br/>DeepAgent + tool convert_currency]
    B[🎫 Booking - 41254<br/>HITL: requires_input]
```

## Bạn đã học được gì?

| Khái niệm | Bạn làm được |
|---|---|
| Agent Card | Khai báo "danh thiếp số" cho agent |
| AgentExecutor | Viết bộ não xử lý yêu cầu (`execute`/`cancel`) |
| TaskUpdater | Báo tiến trình & kết quả cho client |
| A2A Client | Gọi agent từ xa qua `call_agent()` |
| Orchestrator | DeepAgent vừa là Server vừa là Client, điều phối worker |
| **HITL (`input-required`)** | Agent tạm dừng hỏi người dùng xác nhận, rồi tiếp tục khi nhận câu trả lời |

## A2A vs "subagents" trong DeepAgents — khác gì? 🤔

Bạn đã quen với `subagents` trong DeepAgents (agent cha gọi agent con). Hãy so sánh:

| Tiêu chí | Subagents (DeepAgents) | A2A |
|---|---|---|
| Phạm vi | **Trong cùng một tiến trình** | **Qua mạng, giữa các dịch vụ** |
| Framework | Bắt buộc cùng DeepAgents | Bất kỳ framework nào (chuẩn mở) |
| Nơi chạy | Cùng máy/cùng app | Máy khác nhau, công ty khác nhau |
| Giao thức | Gọi hàm nội bộ | HTTP + JSON-RPC |
| Độ tách biệt | Thấp (chung bộ nhớ) | Cao (opaque, không lộ nội bộ) |
| HITL | Có `interrupt()` riêng của LangGraph | Chuẩn chung: trạng thái `input-required` |

> 💡 **Quy tắc chọn:** Việc trong một app → dùng subagents. Việc giữa các app/dịch vụ độc lập → dùng A2A. Trong thực tế người ta **kết hợp cả hai**: DeepAgents lo việc bên trong, A2A lo việc bên ngoài.

## Tài liệu tham khảo trong `refs/`

- `refs/A2A/docs/specification.md` — đặc tả giao thức A2A 1.0
- `refs/A2A/docs/topics/key-concepts.md` — khái niệm cốt lõi
- `refs/A2A/docs/topics/life-of-a-task.md` — vòng đời Task (gồm `input-required`)
- `refs/a2a-python/samples/hello_world_agent.py` — mẫu server chính thức
- `refs/a2a-python/samples/cli.py` — mẫu client chính thức

## Bài tập về nhà 🏠

1. **Đọc thêm:** mở `refs/a2a-python/samples/hello_world_agent.py` và đối chiếu từng dòng với `weather_agent.py` mà bạn vừa viết. Ghi ra 3 điểm giống và 3 điểm khác.
2. **Thêm agent thứ 5:** viết một `translate_agent.py` (chuyên dịch thuật) rồi nối vào orchestrator (thêm tool `ask_translate`). Nhớ chọn port mới!
3. **Bật DeepAgent thật:** set API key (xem Chương 3) rồi chạy lại từ đầu. So sánh câu trả lời MOCK và thật.
4. **Nâng cao HITL:** sửa Booking Agent để hỏi **2 lần** (xác nhận vé → xác nhận thanh toán), hoặc thêm `payment_agent.py` yêu cầu xác nhận thanh toán và nối vào orchestrator.
5. **Nâng cao:** đổi worker sang dùng `InMemoryStore`/checkpointer của DeepAgents để worker có **trí nhớ** qua nhiều lượt hỏi.

# 🧹 Dọn dẹp (chạy cuối buổi học)

Cell này tắt toàn bộ server để giải phóng cổng. Chạy lại từ đầu khi muốn học lại.


In [24]:
stop_all_servers()

Đã dừng tất cả server.
